# Stage 2 - Classical Preprocessing Pipeline v2
## Explainable Hybrid Quantum-Classical NIDS
**Version 2.0 | March 2026**

Key change from v1: SMOTE targets revised to reduce MALWARE synthetic percentage.
Malware: 25K->10K (81% synthetic -> 53%). class_weights.json added as 9th artefact.

| Property | Value |
|---|---|
| SMOTE v2 | RECON->150K, EXPLOIT->95K, MALWARE->10K |
| Output | 9 artefacts (adds stage2_class_weights.json) |
| PCA | NEVER |

In [1]:
!pip install scikit-learn imblearn tqdm --quiet
# ============================================================
# Cell 1 - Environment Setup and Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import json
import os
import logging
import threading
import time
from datetime import datetime, timezone
from collections import OrderedDict

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from tqdm.auto import tqdm

# Pipeline Constants
SCHEMA_VERSION       = "1.0"
QUANTUM_DIM          = 8
RANDOM_STATE         = 42
SENTINEL_VALUE       = -1.0
SENT_CAT             = "__ABSENT__"
TEST_SIZE            = 0.20
OHE_CARD_THRESHOLD   = 10
INT32_MAX            = 2_147_483_647

# REVISED SMOTE TARGETS v2
# Problem: MALWARE original target 25K with ~4.7K real = 81% synthetic
# The VAE could not learn real MALWARE patterns -> latent scatter -> VQC F1=0.04
# Fix: reduce to 2x real count, compensate with class weights downstream
#
#   Class     v1 target   v2 target   Real est.   v1 synth%   v2 synth%
#   Recon     not set     150,000     116,548      0%         22%
#   Exploit   100,000      95,000      47,864      52%        50%
#   Malware    25,000      10,000       4,772      81%        53% <-- KEY FIX
SMOTE_TARGETS = {
    2: 150_000,
    3:  95_000,
    4:  10_000,
}

LABEL_ENCODING = {
    "NORMALL": 0,
    "DoSD":    1,
    "PROBE":   2,
    "EXPLOIT": 3,
    "MALWARE": 4,
}

METADATA_COLS = [
    "source_dataset", "original_label", "master_label",
    "mapping_confidence", "flag_for_review", "quality_pass",
    "quality_flags", "has_missing", "bounds_violation",
    "ingestion_timestamp", "schema_version",
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("Stage2")

STAGE1_DIR = "../DataSets"
OUTPUT_DIR = "."          # legacy (unused after v2 dual-output)

# Dual output directories for VAE-A and VAE-B
OUTPUT_DIR_WITH    = "stage_2_with_zero_v2"     # 167 features — VAE-A input
OUTPUT_DIR_WITHOUT = "stage_2_without_zero_v2"  # ~140 features — VAE-B input

import os as _tmpOS
_tmpOS.makedirs(OUTPUT_DIR_WITH,    exist_ok=True)
_tmpOS.makedirs(OUTPUT_DIR_WITHOUT, exist_ok=True)
del _tmpOS

PARQUET_FILES = {
    "NSL-KDD":     os.path.join(STAGE1_DIR, "nsl_kdd_stage1_output.parquet"),
    "UNSW-NB15":   os.path.join(STAGE1_DIR, "unsw_nb15_stage1_output.parquet"),
    "CICIDS-2017": os.path.join(STAGE1_DIR, "cicids2017_stage1_output.parquet"),
}

ALIGNMENT_MAP_PATH = os.path.join(STAGE1_DIR, "feature_alignment_map.json")
QG03_PATH          = os.path.join(STAGE1_DIR, "qg03_replacement_values.json")

log.info("Stage 2 pipeline v2 initialised.")
print(f"SCHEMA_VERSION = {SCHEMA_VERSION}")
print(f"RANDOM_STATE   = {RANDOM_STATE}")
print(f"SENTINEL_VALUE = {SENTINEL_VALUE}")
print(f"TEST_SIZE      = {TEST_SIZE}")
print(f"SMOTE_TARGETS  = {SMOTE_TARGETS}")
print()
print("SMOTE v2 synthetic % per class:")
_real_est = {2: 116_548, 3: 47_864, 4: 4_772}
_cnames   = {2: "Recon", 3: "Exploit", 4: "Malware"}
for _cls, _tgt in SMOTE_TARGETS.items():
    _real = _real_est[_cls]
    _spct = max(0, (_tgt - _real) / _tgt * 100) if _tgt > _real else 0
    print(f"  {_cnames[_cls]:<10}: {_real:>7,} real -> {_tgt:>7,} target  ({_spct:.1f}% synthetic)")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
07:39:51 | INFO    | Stage 2 pipeline v2 initialised.


SCHEMA_VERSION = 1.0
RANDOM_STATE   = 42
SENTINEL_VALUE = -1.0
TEST_SIZE      = 0.2
SMOTE_TARGETS  = {2: 150000, 3: 95000, 4: 10000}

SMOTE v2 synthetic % per class:
  Recon     : 116,548 real -> 150,000 target  (22.3% synthetic)
  Exploit   :  47,864 real ->  95,000 target  (49.6% synthetic)
  Malware   :   4,772 real ->  10,000 target  (52.3% synthetic)


---
## Step 1 — Load Stage 1 Outputs and Schema Validation

Read the three Stage 1 Parquet files and verify each satisfies the Stage 1 output schema contract.

**Operations:**
1. Read each Parquet file with `pd.read_parquet()`
2. Assert `schema_version` column exists and equals `'1.0'`
3. Assert all 11 mandatory metadata columns are present
4. Log record count, feature column count, and schema version per dataset
5. Load `feature_alignment_map.json` and `qg03_replacement_values.json`

In [2]:
# ============================================================
# Step 1 — Load Stage 1 Outputs and Schema Validation
# ============================================================

# ── 1a. Load JSON artefacts ─────────────────────────────────
with open(ALIGNMENT_MAP_PATH, "r") as f:
    alignment_map = json.load(f)
log.info(f"Loaded feature_alignment_map.json  —  {alignment_map['total_unique_features']} union features")

with open(QG03_PATH, "r") as f:
    qg03_info = json.load(f)
log.info(f"Loaded qg03_replacement_values.json  —  leakage_acknowledged={qg03_info['leakage_acknowledged']}")

# Alias for later use
UNION_FEATURES       = sorted(alignment_map["union_features"])
PER_DATASET_FEATURES = alignment_map["per_dataset_features"]

# ── 1b. Load Parquet files with schema validation ───────────
datasets_raw = {}

for ds_name, path in PARQUET_FILES.items():
    df = pd.read_parquet(path)

    # Validate schema_version
    assert "schema_version" in df.columns, \
        f"{ds_name}: 'schema_version' column missing!"
    version_vals = df["schema_version"].unique()
    assert len(version_vals) == 1 and version_vals[0] == SCHEMA_VERSION, \
        f"{ds_name}: Expected schema_version='{SCHEMA_VERSION}', got {version_vals}"

    # Validate all 11 metadata columns present
    missing_meta = [c for c in METADATA_COLS if c not in df.columns]
    assert len(missing_meta) == 0, \
        f"{ds_name}: Missing metadata columns: {missing_meta}"

    # Separate feature columns from metadata
    feature_cols = [c for c in df.columns if c not in METADATA_COLS]

    datasets_raw[ds_name] = df
    log.info(
        f"  {ds_name:12s} | records={len(df):>10,} | "
        f"feature_cols={len(feature_cols):>3} | schema_version={version_vals[0]}"
    )

total_records = sum(len(df) for df in datasets_raw.values())
log.info(f"  COMBINED     | records={total_records:>10,}")
print(f"\n{'='*60}")
print(f"Stage 1 loaded successfully.  Total records: {total_records:,}")
print(f"{'='*60}")

07:40:25 | INFO    | Loaded feature_alignment_map.json  —  154 union features


07:40:25 | INFO    | Loaded qg03_replacement_values.json  —  leakage_acknowledged=True
07:40:26 | INFO    |   NSL-KDD      | records=   147,907 | feature_cols= 36 | schema_version=1.0
07:40:27 | INFO    |   UNSW-NB15    | records=   198,802 | feature_cols= 42 | schema_version=1.0
07:40:32 | INFO    |   CICIDS-2017  | records= 2,522,326 | feature_cols= 77 | schema_version=1.0
07:40:32 | INFO    |   COMBINED     | records= 2,869,035



Stage 1 loaded successfully.  Total records: 2,869,035


---
## Step 2 — Feature Alignment

Unify the three datasets into a single feature matrix where every record has exactly the same 154 columns.

**Sentinel Policy (D7-1):**
| Feature Type | Sentinel Value | Post-scaling Value |
|---|---|---|
| Numeric | `-1.0` | `0.0` |
| Categorical (string) | `'__ABSENT__'` | `0.0` (all-zeros OHE or freq=0) |

After padding, all DataFrames are forced to the same **sorted** column order before concatenation.

In [3]:

# ============================================================
# Step 2 — Feature Alignment   (chunked stream → parquet)
# ============================================================
# Root-cause of prior MemoryErrors:
#   feat = df[feature_cols].copy()  →  1 GB copy of the CICIDS-2017 block
#   pd.concat(aligned_dfs)         →  holds all 3 aligned DFs simultaneously
#
# Fix — process each dataset in ROW CHUNKS of 500 K:
#   • Each chunk: ~154 cols × 500 K rows × 4 B = ~300 MB
#   • Source df stays in RAM (~1 GB) but no full feature-block copy is made.
#   • Each chunk is written to parquet immediately, then discarded.
#   • Peak per-iteration ≈ source_df + 1 chunk ≈ 1.3 GB (well within 2.5 GB limit).

import gc           as _gc
import pyarrow      as _pa
import pyarrow.parquet as _pq

_UNIFIED_PARQUET = os.path.join(OUTPUT_DIR, "_unified_aligned.parquet")
_CHUNK_ALIGN     = 500_000   # rows per aligned chunk

CATEGORICAL_COLS_PER_DS = {
    "nsl_kdd":    ["protocol_type", "service", "flag"],
    "unsw_nb15":  ["proto", "state", "service"],
    "cicids2017": [],
}
ALL_CATEGORICAL_COLS = set()
for _cols in CATEGORICAL_COLS_PER_DS.values():
    ALL_CATEGORICAL_COLS.update(_cols)
ALL_CATEGORICAL_COLS = sorted(ALL_CATEGORICAL_COLS)
log.info(f"Categorical columns across all datasets: {ALL_CATEGORICAL_COLS}")

_DS_KEY_MAP = {
    "NSL-KDD":     "nsl_kdd",
    "UNSW-NB15":   "unsw_nb15",
    "CICIDS-2017": "cicids2017",
}

def _align_chunk(chunk, union_features, meta_cols, all_cat_cols_set):
    """
    Align a row-slice of one dataset to the union feature schema.
    Returns a pd.DataFrame with columns = meta_cols + union_features, all
    numeric features as float32.  Never copies the entire block at once:
    each column is extracted individually (≤ 4 MB per column for 500 K rows).
    """
    meta_set = set(meta_cols)
    out = {}

    # ① Metadata columns (small; fine to copy individually)
    for c in meta_cols:
        out[c] = chunk[c].values.copy()

    # ② Feature columns — one column at a time to avoid a full-block copy
    for col in union_features:
        if col in chunk.columns and col not in meta_set:
            if col not in all_cat_cols_set and pd.api.types.is_numeric_dtype(chunk[col]):
                out[col] = chunk[col].values.astype(np.float32)   # cast only this col
            else:
                out[col] = chunk[col].values.copy()
        else:
            # Absent in this dataset → fill with sentinel (scalar; pandas broadcasts)
            out[col] = SENT_CAT if col in all_cat_cols_set else np.float32(SENTINEL_VALUE)

    return pd.DataFrame(out)   # absent-column scalars expanded here (~200 MB for 500 K rows)


# ── Stream each dataset to parquet in CHUNKS ────────────────
# pq_state[0] = writer, pq_state[1] = reference schema
_pq_state  = [None, None]
_all_cat_set = set(ALL_CATEGORICAL_COLS)

for _ds_name in tqdm(list(datasets_raw.keys()),
                     desc="Step 2 — align & write", unit="dataset"):

    _df      = datasets_raw.pop(_ds_name)      # remove from dict → freed when done
    _n_rows  = len(_df)
    _own_f   = set(PER_DATASET_FEATURES[_DS_KEY_MAP[_ds_name]])
    _n_abs   = len(UNION_FEATURES) - len(_own_f)
    log.info(f"  {_ds_name:12s} | own={len(_own_f):>3} | absent={_n_abs:>3} | "
             f"sentinel_rate={_n_abs/len(UNION_FEATURES)*100:.1f}% | rows={_n_rows:,}")

    for _start in tqdm(range(0, _n_rows, _CHUNK_ALIGN),
                       desc=f"  chunks ({_ds_name})", leave=False, unit="chunk"):
        _chunk_df  = _align_chunk(_df.iloc[_start : _start + _CHUNK_ALIGN],
                                  UNION_FEATURES, METADATA_COLS, _all_cat_set)
        _tbl       = _pa.Table.from_pandas(_chunk_df, preserve_index=False)
        del _chunk_df
        _gc.collect()

        if _pq_state[0] is None:
            _pq_state[1] = _tbl.schema
            _pq_state[0] = _pq.ParquetWriter(_UNIFIED_PARQUET, _pq_state[1])
        elif not _tbl.schema.equals(_pq_state[1], check_metadata=False):
            _tbl = _tbl.cast(_pq_state[1])

        _pq_state[0].write_table(_tbl)
        del _tbl
        _gc.collect()

    del _df
    _gc.collect()

if _pq_state[0]:
    _pq_state[0].close()

# ── Read unified DataFrame from the merged parquet ──────────
unified_df = pd.read_parquet(_UNIFIED_PARQUET)

feature_cols_unified = [c for c in unified_df.columns if c not in METADATA_COLS]
assert len(feature_cols_unified) == 154, \
    f"Expected 154 feature columns, got {len(feature_cols_unified)}"

log.info(f"Unified DataFrame: {unified_df.shape[0]:,} records × {len(feature_cols_unified)} features")
print(f"\nUnified shape: {unified_df.shape}")
print(f"Feature columns: {len(feature_cols_unified)}")
print(f"\nClass distribution:")
print(unified_df["master_label"].value_counts().to_string())


07:40:40 | INFO    | Categorical columns across all datasets: ['flag', 'proto', 'protocol_type', 'service', 'state']


Step 2 — align & write:   0%|          | 0/3 [00:00<?, ?dataset/s]

07:40:40 | INFO    |   NSL-KDD      | own= 36 | absent=118 | sentinel_rate=76.6% | rows=147,907


  chunks (NSL-KDD):   0%|          | 0/1 [00:00<?, ?chunk/s]

07:40:42 | INFO    |   UNSW-NB15    | own= 42 | absent=112 | sentinel_rate=72.7% | rows=198,802


  chunks (UNSW-NB15):   0%|          | 0/1 [00:00<?, ?chunk/s]

07:40:45 | INFO    |   CICIDS-2017  | own= 77 | absent= 77 | sentinel_rate=50.0% | rows=2,522,326


  chunks (CICIDS-2017):   0%|          | 0/6 [00:00<?, ?chunk/s]

07:41:20 | INFO    | Unified DataFrame: 2,869,035 records × 154 features



Unified shape: (2869035, 165)
Feature columns: 154

Class distribution:
master_label
NORMALL    2266451
DoSD        391102
PROBE       145683
EXPLOIT      59832
MALWARE       5967


---
## Step 3 — Dual-Stratified Train / Test Split

The train/test split **must** occur before any encoder or scaler is fitted. This is the most important leakage-prevention step.

**Stratification**: Compound key `master_label + '__' + source_dataset` creates 15 strata, ensuring both class distribution and dataset provenance are proportionally preserved.

| Parameter | Value |
|---|---|
| Splitter | `StratifiedShuffleSplit` |
| Test size | 0.20 |
| Random state | 42 |
| Stratify on | `master_label__source_dataset` |

In [4]:
# ============================================================
# Step 3 — Dual-Stratified Train / Test Split
# ============================================================

# ── 3a. Encode master_label to integer ──────────────────────
unified_df["y"] = unified_df["master_label"].map(LABEL_ENCODING)
assert unified_df["y"].notna().all(), "Unmapped labels found!"

# ── 3b. Build compound stratification key ───────────────────
strat_key = unified_df["master_label"] + "__" + unified_df["source_dataset"]
n_strata = strat_key.nunique()
log.info(f"Stratification strata: {n_strata}")
print(f"Strata ({n_strata}):")
print(strat_key.value_counts().sort_index().to_string())

# ── 3c. Perform the split ──────────────────────────────────
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(splitter.split(unified_df, strat_key))

# Separate features, labels, and metadata
feature_cols = [c for c in unified_df.columns if c not in METADATA_COLS + ["y"]]

X_train_raw = unified_df.iloc[train_idx][feature_cols].reset_index(drop=True)
X_test_raw  = unified_df.iloc[test_idx][feature_cols].reset_index(drop=True)
y_train     = unified_df.iloc[train_idx]["y"].values.copy()
y_test      = unified_df.iloc[test_idx]["y"].values.copy()

# Preserve source_dataset for later steps (sentinel mask, clipping)
meta_train = unified_df.iloc[train_idx][METADATA_COLS].reset_index(drop=True)
meta_test  = unified_df.iloc[test_idx][METADATA_COLS].reset_index(drop=True)

log.info(f"Train: {X_train_raw.shape[0]:>10,} records ({X_train_raw.shape[0]/len(unified_df)*100:.1f}%)")
log.info(f"Test:  {X_test_raw.shape[0]:>10,} records ({X_test_raw.shape[0]/len(unified_df)*100:.1f}%)")

# ── 3d. Log split distribution ──────────────────────────────
print(f"\n{'='*60}")
print(f"Train shape: {X_train_raw.shape}")
print(f"Test  shape: {X_test_raw.shape}")
print(f"\nTrain class distribution:")
labels_inv = {v: k for k, v in LABEL_ENCODING.items()}
for lbl in sorted(np.unique(y_train)):
    cnt = (y_train == lbl).sum()
    print(f"  {labels_inv[lbl]:8s} ({lbl}): {cnt:>10,}  ({cnt/len(y_train)*100:.2f}%)")
print(f"\nTest class distribution:")
for lbl in sorted(np.unique(y_test)):
    cnt = (y_test == lbl).sum()
    print(f"  {labels_inv[lbl]:8s} ({lbl}): {cnt:>10,}  ({cnt/len(y_test)*100:.2f}%)")

# Log flag_for_review counts (FR-S1-07)
for split_name, meta_split in [("Train", meta_train), ("Test", meta_test)]:
    review_count = meta_split["flag_for_review"].sum()
    log.info(f"  {split_name} flag_for_review count: {review_count}")

# Free large unified_df
del unified_df
import gc; gc.collect()
print(f"{'='*60}")

07:41:39 | INFO    | Stratification strata: 14


Strata (14):
DoSD__CICIDS-2017        321764
DoSD__NSL-KDD             52985
DoSD__UNSW-NB15           16353
EXPLOIT__CICIDS-2017      11306
EXPLOIT__NSL-KDD           4001
EXPLOIT__UNSW-NB15        44525
MALWARE__CICIDS-2017       1953
MALWARE__UNSW-NB15         4014
NORMALL__CICIDS-2017    2096484
NORMALL__NSL-KDD          76967
NORMALL__UNSW-NB15        93000
PROBE__CICIDS-2017        90819
PROBE__NSL-KDD            13954
PROBE__UNSW-NB15          40910


07:42:22 | INFO    | Train:  2,295,228 records (80.0%)
07:42:22 | INFO    | Test:     573,807 records (20.0%)
07:42:22 | INFO    |   Train flag_for_review count: 0
07:42:22 | INFO    |   Test flag_for_review count: 0



Train shape: (2295228, 154)
Test  shape: (573807, 154)

Train class distribution:
  NORMALL  (0):  1,813,161  (79.00%)
  DoSD     (1):    312,881  (13.63%)
  PROBE    (2):    116,546  (5.08%)
  EXPLOIT  (3):     47,866  (2.09%)
  MALWARE  (4):      4,774  (0.21%)

Test class distribution:
  NORMALL  (0):    453,290  (79.00%)
  DoSD     (1):     78,221  (13.63%)
  PROBE    (2):     29,137  (5.08%)
  EXPLOIT  (3):     11,966  (2.09%)
  MALWARE  (4):      1,193  (0.21%)


---
## Step 4 — Categorical Encoding

Two encoding strategies based on column cardinality (threshold = 10 unique real values):
- **≤ 10 unique**: One-Hot Encoding (OHE)
- **> 10 unique**: FrequencyEncoder (custom)

All encoders fitted **exclusively** on the training split.

| Column | Dataset(s) | Method |
|---|---|---|
| `protocol_type` | NSL-KDD | OHE |
| `flag` | NSL-KDD | Frequency |
| `service` | NSL-KDD + UNSW-NB15 | Frequency |
| `proto` | UNSW-NB15 | Frequency |
| `state` | UNSW-NB15 | OHE or Frequency (runtime) |

In [5]:

# ============================================================
# Step 4 — Categorical Encoding
# ============================================================

# ── 4a. FrequencyEncoder class ──────────────────────────────
class FrequencyEncoder:
    """
    Maps each category to its relative frequency in the training set.
    __ABSENT__ → SENTINEL_VALUE (-1.0)
    Unseen test values → 0.0
    """
    def __init__(self):
        self.freq_maps = {}   # col -> {value: freq}

    def fit(self, df, columns):
        for col in columns:
            real_vals = df.loc[df[col] != SENT_CAT, col]
            freq_map = real_vals.value_counts(normalize=True).to_dict()
            self.freq_maps[col] = freq_map
            log.info(f"  FreqEnc fit '{col}': {len(freq_map)} unique real values")
        return self

    def transform(self, df, columns):
        df = df.copy()
        for col in columns:
            freq_map = self.freq_maps[col]
            def _map_val(v):
                if v == SENT_CAT:
                    return SENTINEL_VALUE
                return freq_map.get(v, 0.0)
            df[col] = df[col].apply(_map_val)
        return df


# ── 4b. Determine encoding method per categorical column ────
ohe_cols  = []
freq_cols = []

for col in ALL_CATEGORICAL_COLS:
    if col not in X_train_raw.columns:
        continue
    # Count unique non-sentinel values in TRAINING set only
    real_unique = X_train_raw.loc[X_train_raw[col] != SENT_CAT, col].nunique()
    if real_unique <= OHE_CARD_THRESHOLD:
        ohe_cols.append(col)
        log.info(f"  {col}: {real_unique} unique → OHE")
    else:
        freq_cols.append(col)
        log.info(f"  {col}: {real_unique} unique → FrequencyEncoder")

# CF-S2-02 FIX: Explicit assertion that 'flag' is FrequencyEncoded (11 unique > threshold 10)
if "flag" in [c for c in ALL_CATEGORICAL_COLS if c in X_train_raw.columns]:
    assert "flag" in freq_cols, (
        f"'flag' should be FrequencyEncoded (expected >10 unique), "
        f"but was assigned to OHE. Check training split cardinality."
    )
    log.info("CF-S2-02: Confirmed 'flag' assigned to freq_cols as expected.")

log.info(f"Final OHE columns:       {ohe_cols}")
log.info(f"Final Frequency columns: {freq_cols}")
print(f"OHE columns:       {ohe_cols}")
print(f"Frequency columns: {freq_cols}")

# ── 4c. Fit and transform FrequencyEncoder ──────────────────
freq_encoder = FrequencyEncoder()
freq_encoder.fit(X_train_raw, freq_cols)

X_train_enc = freq_encoder.transform(X_train_raw, freq_cols)
X_test_enc  = freq_encoder.transform(X_test_raw, freq_cols)

# ── 4d. Fit and transform OHE ──────────────────────────────
# Build feature lineage for sentinel mask propagation
feature_lineage = {}  # new_col_name -> parent_col_name

if ohe_cols:
    # FIX: Use a guaranteed out-of-vocabulary placeholder for sentinel rows.
    # The previous approach (dropping all rows where ANY column == SENT_CAT) fails
    # with "ValueError: 0 samples" when every row has at least one sentinel column.
    # Instead, replace sentinels with a placeholder and fit on ALL rows.
    # handle_unknown='ignore' ensures __OHE_ABSENT__ → all-zeros, preserving the
    # sentinel → all-zero output behaviour exactly.
    _OHE_ABSENT_PLACEHOLDER = "__OHE_ABSENT__"

    ohe = OneHotEncoder(
        sparse_output=False,
        handle_unknown="ignore",
        dtype=np.float64,
    )

    train_ohe_data = X_train_enc[ohe_cols].copy()
    train_ohe_fit = train_ohe_data.copy()
    for c in ohe_cols:
        train_ohe_fit.loc[train_ohe_fit[c] == SENT_CAT, c] = _OHE_ABSENT_PLACEHOLDER

    ohe.fit(train_ohe_fit)

    # Store OHE categories for artefact
    ohe_categories = {}
    for i, col in enumerate(ohe_cols):
        ohe_categories[col] = ohe.categories_[i].tolist()
        log.info(f"  OHE '{col}' categories: {ohe.categories_[i].tolist()}")

    # Transform function that handles sentinels
    def apply_ohe(df, ohe_model, ohe_columns):
        ohe_data = df[ohe_columns].copy()
        sentinel_mask_ohe = (ohe_data == SENT_CAT).any(axis=1)

        # Replace sentinels with guaranteed-absent placeholder
        ohe_data_filled = ohe_data.copy()
        for c in ohe_columns:
            ohe_data_filled.loc[ohe_data_filled[c] == SENT_CAT, c] = _OHE_ABSENT_PLACEHOLDER

        encoded = ohe_model.transform(ohe_data_filled)

        # Force sentinel rows to all-zeros (safety belt)
        encoded[sentinel_mask_ohe.values] = 0.0

        # Build column names
        encoded_col_names = []
        for i, col in enumerate(ohe_columns):
            for cat in ohe_model.categories_[i]:
                encoded_col_names.append(f"{col}__{cat}")

        encoded_df = pd.DataFrame(encoded, columns=encoded_col_names, index=df.index)
        return encoded_df, encoded_col_names

    # Apply OHE to train
    ohe_train_df, ohe_col_names = apply_ohe(X_train_enc, ohe, ohe_cols)
    ohe_test_df, _              = apply_ohe(X_test_enc,  ohe, ohe_cols)

    # Build feature lineage: map each OHE-derived column to its parent
    for i, col in enumerate(ohe_cols):
        for cat in ohe.categories_[i]:
            feature_lineage[f"{col}__{cat}"] = col

    # FIX-1: Drop original OHE columns and insert encoded ones (INSIDE if ohe_cols block)
    X_train_enc = X_train_enc.drop(columns=ohe_cols)
    X_test_enc  = X_test_enc.drop(columns=ohe_cols)
    X_train_enc = pd.concat([X_train_enc, ohe_train_df], axis=1)
    X_test_enc  = pd.concat([X_test_enc, ohe_test_df], axis=1)

else:
    ohe_categories = {}
    ohe_col_names = []

# ── 4e. Ensure all values are numeric ──────────────────────
# Check no string values remain
for col in X_train_enc.columns:
    if X_train_enc[col].dtype == object:
        raise ValueError(f"Column '{col}' is still object type after encoding!")

X_train_enc = X_train_enc.astype(np.float64)
X_test_enc  = X_test_enc.astype(np.float64)

# Record final feature names (post-encoding, deterministic order)
post_enc_feature_names = sorted(X_train_enc.columns.tolist())
X_train_enc = X_train_enc[post_enc_feature_names]
X_test_enc  = X_test_enc[post_enc_feature_names]

n_features_post_enc = len(post_enc_feature_names)

log.info(f"Post-encoding feature count: {n_features_post_enc} (was 154 pre-encoding)")
print(f"\nPost-encoding columns: {n_features_post_enc}")
print(f"OHE columns added: {len(ohe_col_names)}")
print(f"Train shape: {X_train_enc.shape}")
print(f"Test  shape: {X_test_enc.shape}")


07:42:41 | INFO    |   flag: 11 unique → FrequencyEncoder
07:42:41 | INFO    |   proto: 133 unique → FrequencyEncoder
07:42:41 | INFO    |   protocol_type: 3 unique → OHE
07:42:41 | INFO    |   service: 79 unique → FrequencyEncoder
07:42:41 | INFO    |   state: 10 unique → OHE
07:42:41 | INFO    | CF-S2-02: Confirmed 'flag' assigned to freq_cols as expected.
07:42:41 | INFO    | Final OHE columns:       ['protocol_type', 'state']
07:42:41 | INFO    | Final Frequency columns: ['flag', 'proto', 'service']
07:42:41 | INFO    |   FreqEnc fit 'flag': 11 unique real values
07:42:41 | INFO    |   FreqEnc fit 'proto': 133 unique real values


OHE columns:       ['protocol_type', 'state']
Frequency columns: ['flag', 'proto', 'service']


07:42:41 | INFO    |   FreqEnc fit 'service': 79 unique real values
07:42:49 | INFO    |   OHE 'protocol_type' categories: ['__OHE_ABSENT__', 'icmp', 'tcp', 'udp']
07:42:49 | INFO    |   OHE 'state' categories: ['ACC', 'CLO', 'CON', 'ECO', 'FIN', 'INT', 'PAR', 'REQ', 'RST', '__OHE_ABSENT__', 'no']
07:42:58 | INFO    | Post-encoding feature count: 167 (was 154 pre-encoding)



Post-encoding columns: 167
OHE columns added: 15
Train shape: (2295228, 167)
Test  shape: (573807, 167)


---
## Step 5 — Sentinel Mask Creation

Boolean array of shape `(N × F)` where `True` = feature absent for that record's dataset.

**Critical**: Must be created **BEFORE** scaling. Built from metadata (`source_dataset`) and feature lineage — no feature values are inspected.

| Stage | How Sentinel Mask is Used |
|---|---|
| Step 9 (Scaling) | Exclude sentinel positions from min/max computation |
| Step 9 (Post-scale) | Zero out sentinel positions after scaling |
| Stage 3 (VAE) | Mask per-feature reconstruction loss |
| Stage 5 (Ensemble) | Auxiliary feature or ignored |

In [6]:
# ============================================================
# Step 5 — Sentinel Mask Creation
# ============================================================

# Map dataset keys to their feature sets (post-encoding column names)
ds_key_to_alignment = {
    "NSL-KDD":     "nsl_kdd",
    "UNSW-NB15":   "unsw_nb15",
    "CICIDS-2017": "cicids2017",
}

def build_sentinel_mask(X_df, meta_df, post_enc_features, per_ds_features, feature_lineage):
    """
    Build boolean sentinel mask: True = feature absent for that record's dataset.
    Uses metadata (source_dataset) and feature lineage, NOT feature values.
    """
    mask = np.zeros((len(X_df), len(post_enc_features)), dtype=bool)

    for ds_name in ["NSL-KDD", "UNSW-NB15", "CICIDS-2017"]:
        ds_key = ds_key_to_alignment[ds_name]
        own_pre_enc = set(per_ds_features[ds_key])

        # Determine which post-encoding columns are ABSENT for this dataset
        absent_col_indices = []
        for j, col_name in enumerate(post_enc_features):
            # Check if this is an OHE-derived column
            parent_col = feature_lineage.get(col_name, col_name)
            if parent_col not in own_pre_enc:
                absent_col_indices.append(j)

        # Find rows belonging to this dataset
        row_mask = (meta_df["source_dataset"] == ds_name).values
        for j in absent_col_indices:
            mask[row_mask, j] = True

    return mask


sentinel_mask_train = build_sentinel_mask(
    X_train_enc, meta_train, post_enc_feature_names,
    PER_DATASET_FEATURES, feature_lineage
)
sentinel_mask_test = build_sentinel_mask(
    X_test_enc, meta_test, post_enc_feature_names,
    PER_DATASET_FEATURES, feature_lineage
)

# ── Verification: sentinel positions should have sentinel values ──
train_vals = X_train_enc.values
for ds_name in ["NSL-KDD", "UNSW-NB15", "CICIDS-2017"]:
    ds_rows = (meta_train["source_dataset"] == ds_name).values
    ds_mask = sentinel_mask_train[ds_rows]
    ds_vals = train_vals[ds_rows]
    sentinel_positions = ds_vals[ds_mask]
    n_sentinel = ds_mask.sum()
    n_correct = np.sum(sentinel_positions == SENTINEL_VALUE)
    log.info(
        f"  {ds_name:12s} | sentinel_positions={n_sentinel:>12,} | "
        f"correct_sentinel_val={n_correct:>12,}"
    )

log.info(f"Sentinel mask train shape: {sentinel_mask_train.shape}")
log.info(f"Sentinel mask test  shape: {sentinel_mask_test.shape}")
print(f"\nSentinel mask created.")
print(f"  Train: {sentinel_mask_train.shape} — {sentinel_mask_train.sum():,} True positions")
print(f"  Test:  {sentinel_mask_test.shape}  — {sentinel_mask_test.sum():,} True positions")

07:43:34 | INFO    |   NSL-KDD      | sentinel_positions=  15,145,728 | correct_sentinel_val=  13,844,142
07:43:36 | INFO    |   UNSW-NB15    | sentinel_positions=  18,289,715 | correct_sentinel_val=  17,653,551
07:43:47 | INFO    |   CICIDS-2017  | sentinel_positions= 181,607,490 | correct_sentinel_val= 151,339,575
07:43:47 | INFO    | Sentinel mask train shape: (2295228, 167)
07:43:47 | INFO    | Sentinel mask test  shape: (573807, 167)



Sentinel mask created.
  Train: (2295228, 167) — 215,042,933 True positions
  Test:  (573807, 167)  — 53,760,733 True positions


---
## Step 6 — CICFlowMeter Integer-Overflow Column Clipping

CICFlowMeter has a known integer overflow bug affecting several columns. Records are **NOT excluded** — the overflow is an instrument artefact. Clipping applied to **CICIDS-2017 rows only**.

| Column | Lower Clip | Upper Clip |
|---|---|---|
| `Fwd Header Length` | 0 | INT32_MAX |
| `Bwd Header Length` | 0 | INT32_MAX |
| `Fwd Header Length.1` | 0 | INT32_MAX |
| `min_seg_size_forward` | 0 | INT32_MAX |
| `Init_Win_bytes_forward` | -1 | 65,535 |
| `Init_Win_bytes_backward` | -1 | 65,535 |
| `Flow Duration` | 0 | No upper |
| `Flow Bytes/s` | 0 | No upper |
| `Flow Packets/s` | 0 | No upper |
| `Flow IAT Mean` | 0 | No upper |

In [7]:

# ============================================================
# Step 6 — CICFlowMeter Integer-Overflow Column Clipping
# ============================================================

# Clip configuration: (column_name, lower_bound, upper_bound)
# None means no clip on that side
CLIP_CONFIG = [
    ("Fwd Header Length",       0,    INT32_MAX),
    ("Bwd Header Length",       0,    INT32_MAX),
    ("Fwd Header Length.1",     0,    INT32_MAX),
    ("min_seg_size_forward",    0,    INT32_MAX),
    ("Init_Win_bytes_forward",  -1,   65535),
    ("Init_Win_bytes_backward", -1,   65535),
    ("Flow Duration",           0,    None),
    ("Flow Bytes/s",            0,    None),
    ("Flow Packets/s",          0,    None),
    ("Flow IAT Mean",           0,    None),
]

def clip_cicflowmeter(X_df, meta_df, clip_config, feature_names):
    """
    Apply clipping to CICIDS-2017 rows only.

    Memory strategy:
      - Do NOT copy the full DataFrame (167 cols × 2.3M rows = 2.86 GB).
      - Copy only the ≤10 clip-target columns (~18 MB each) one at a time.
      - Build the output by assembling: modified columns (new arrays) +
        all other columns as zero-copy references to the input arrays.
      Peak extra RAM ≈ 10 × 18 MB = ~180 MB.
    """
    cic_rows = (meta_df["source_dataset"] == "CICIDS-2017").values
    n_clipped_total = 0

    # 1. Process each clip column; store modified arrays in a dict.
    clip_col_set = {cfg[0] for cfg in clip_config}
    modified = {}   # col_name → new 1-D numpy array (full-length)

    for col_name, lower, upper in clip_config:
        if col_name not in feature_names:
            log.info(f"  Skipping '{col_name}' — not in post-encoding features")
            continue
        if col_name not in X_df.columns:
            log.info(f"  Skipping '{col_name}' — not a column in X_df")
            continue

        # Copy just this one column (~18 MB) — never the whole array.
        col_vals = X_df[col_name].to_numpy(dtype=np.float32, copy=True)
        cic_slice = col_vals[cic_rows].copy()

        if lower is not None:
            np.clip(cic_slice, a_min=lower, a_max=None, out=cic_slice)
        if upper is not None:
            np.clip(cic_slice, a_min=None, a_max=upper, out=cic_slice)

        n_changed = int(np.sum(X_df[col_name].to_numpy()[cic_rows] != cic_slice))
        n_clipped_total += n_changed
        col_vals[cic_rows] = cic_slice
        modified[col_name] = col_vals

        if n_changed > 0:
            log.info(f"  Clipped '{col_name}': {n_changed:,} values changed")

    # 2. Build output DataFrame: reference original column data for untouched
    #    columns (no copy), use the new arrays for clipped columns.
    out_data = {
        col: (modified[col] if col in modified else X_df[col].to_numpy(copy=False))
        for col in X_df.columns
    }
    X_out = pd.DataFrame(out_data, index=X_df.index, copy=False)
    return X_out, n_clipped_total


X_train_clipped, n_clip_train = clip_cicflowmeter(
    X_train_enc, meta_train, CLIP_CONFIG, post_enc_feature_names
)
X_test_clipped, n_clip_test = clip_cicflowmeter(
    X_test_enc, meta_test, CLIP_CONFIG, post_enc_feature_names
)

log.info(f"Total values clipped — Train: {n_clip_train:,} | Test: {n_clip_test:,}")
print(f"\nClipping complete.")
print(f"  Train: {n_clip_train:,} values clipped")
print(f"  Test:  {n_clip_test:,} values clipped")


07:43:56 | INFO    |   Clipped 'Fwd Header Length': 24 values changed
07:43:56 | INFO    |   Clipped 'Bwd Header Length': 15 values changed
07:43:56 | INFO    |   Clipped 'Fwd Header Length.1': 24 values changed
07:43:56 | INFO    |   Clipped 'min_seg_size_forward': 24 values changed
07:43:56 | INFO    |   Clipped 'Flow Duration': 87 values changed
07:43:56 | INFO    |   Clipped 'Flow Bytes/s': 355 values changed
07:43:57 | INFO    |   Clipped 'Flow Packets/s': 87 values changed
07:43:57 | INFO    |   Clipped 'Flow IAT Mean': 87 values changed
07:43:57 | INFO    |   Clipped 'Fwd Header Length': 11 values changed
07:43:57 | INFO    |   Clipped 'Bwd Header Length': 7 values changed
07:43:57 | INFO    |   Clipped 'Fwd Header Length.1': 11 values changed
07:43:57 | INFO    |   Clipped 'min_seg_size_forward': 11 values changed
07:43:57 | INFO    |   Clipped 'Flow Duration': 20 values changed
07:43:57 | INFO    |   Clipped 'Flow Bytes/s': 76 values changed
07:43:57 | INFO    |   Clipped 'Flo


Clipping complete.
  Train: 703 values clipped
  Test:  176 values clipped


---
## Step 7 — Imputation for True Missing Values

After alignment and sentinel padding, the only remaining NaN values are **1,358 CICIDS-2017 records** where `Flow Bytes/s` is genuinely missing.

- **Strategy**: Median imputation (`SimpleImputer`, `strategy='median'`)
- **Fit on**: Training set only
- **Post-imputation**: Hard assert that no NaN remains

In [8]:

# ============================================================
# Step 7 — Imputation for True Missing Values
# ============================================================

# ── 7a. Check NaN before imputation ────────────────────────
nan_train = X_train_clipped.isna().sum()
nan_test  = X_test_clipped.isna().sum()

nan_cols_train = nan_train[nan_train > 0]
nan_cols_test  = nan_test[nan_test > 0]

print("NaN counts BEFORE imputation:")
print(f"  Train: {nan_cols_train.sum()} NaN across {len(nan_cols_train)} column(s)")
if len(nan_cols_train) > 0:
    print(f"    {nan_cols_train.to_dict()}")
print(f"  Test:  {nan_cols_test.sum()} NaN across {len(nan_cols_test)} column(s)")
if len(nan_cols_test) > 0:
    print(f"    {nan_cols_test.to_dict()}")

# ── 7b. Targeted median imputation (NaN columns only) ───────
# SimpleImputer / np.isnan(df.values) on the full matrix (~2.3 M × 167) builds
# a 2.86 GiB intermediate array and causes a MemoryError.  Since only a small
# number of columns actually contain NaN, we compute medians for those columns
# only and fill with DataFrame.fillna, which is column-wise and memory-efficient.

nan_cols_list = nan_cols_train.index.tolist()   # columns with NaN in train

# Compute column medians on training set only (column-wise, no full copy)
fill_values = {col: float(X_train_clipped[col].median()) for col in nan_cols_list}

log.info(f"Imputation fill values (train medians): {fill_values}")
print(f"\nImputation fill values (from training medians): {fill_values}")

X_train_imputed = X_train_clipped.fillna(fill_values)
X_test_imputed  = X_test_clipped.fillna(fill_values)

# ── 7c. Hard assert: no NaN remaining ──────────────────────
# Use pandas column-wise isna() to avoid allocating a full numpy matrix copy
assert not X_train_imputed.isna().any().any(), \
    "NaN found in training set after imputation!"
assert not X_test_imputed.isna().any().any(), \
    "NaN found in test set after imputation!"

log.info("Imputation complete. Zero NaN in train and test.")
print(f"Post-imputation: 0 NaN in train, 0 NaN in test ✓")


NaN counts BEFORE imputation:
  Train: 290 NaN across 1 column(s)
    {'Flow Bytes/s': 290}
  Test:  63 NaN across 1 column(s)
    {'Flow Bytes/s': 63}


07:44:20 | INFO    | Imputation fill values (train medians): {'Flow Bytes/s': 1730.55517578125}



Imputation fill values (from training medians): {'Flow Bytes/s': 1730.55517578125}


07:44:22 | INFO    | Imputation complete. Zero NaN in train and test.


Post-imputation: 0 NaN in train, 0 NaN in test ✓


---
## Step 8 — QG-03 Leakage Documentation

The QG-03 Infinity replacement values were computed on the full CICIDS-2017 dataset (train+test combined). This is an **acknowledged minor leakage** affecting 0.002% of CICIDS-2017 values.

**Operations:**
1. Load `qg03_replacement_values.json` and log the leakage acknowledgement
2. Verify **no Infinity values** remain in train or test
3. If Infinity found → halt execution

In [9]:

# ============================================================
# Step 8 — QG-03 Leakage Documentation
# ============================================================

# ── 8a. Log leakage acknowledgement ────────────────────────
print("QG-03 Leakage Documentation")
print("=" * 60)
print(f"WARNING: {qg03_info['WARNING']}")
print(f"Leakage acknowledged: {qg03_info['leakage_acknowledged']}")
print(f"Computation scope:    {qg03_info['computation_scope']}")
print()

for ds_key, replacements in [("nsl_kdd", "NSL-KDD"), ("unsw_nb15", "UNSW-NB15"), ("cicids2017", "CICIDS-2017")]:
    ds_info = qg03_info.get(ds_key, {})
    if ds_info:
        print(f"  {replacements}:")
        for col, details in ds_info.items():
            print(f"    {col}: {details['count']} values replaced with {details['replacement']}")
    else:
        print(f"  {replacements}: No replacements")

# ── 8b. Verify no Infinity values remain ───────────────────
# Avoid .values on the full DataFrame (allocates ~2.86 GiB).
# np.isinf is equivalent to (x == +inf) | (x == -inf); replicate column-wise.
def _has_inf(df):
    """Check for ±inf without materialising the full numpy matrix."""
    pos_inf = np.float64(np.inf)
    neg_inf = np.float64(-np.inf)
    for col in df.columns:
        s = df[col]
        if (s == pos_inf).any() or (s == neg_inf).any():
            return True
    return False

has_inf_train = _has_inf(X_train_imputed)
has_inf_test  = _has_inf(X_test_imputed)

assert not has_inf_train, \
    "CRITICAL: Infinity values found in training set! Stage 1 QG-03 may not have been applied."
assert not has_inf_test, \
    "CRITICAL: Infinity values found in test set! Stage 1 QG-03 may not have been applied."

log.info("QG-03 verification passed: No Infinity values in train or test.")
print(f"\nInfinity check: PASSED ✓")
print(f"  Train: has_inf = {has_inf_train}")
print(f"  Test:  has_inf = {has_inf_test}")
print(f"\nLeakage quantification:")
cic_total_values = 2_830_743 * 77  # approximate CICIDS-2017 records × features
affected_values = 1509 + 2867
print(f"  Affected: {affected_values:,} / {cic_total_values:,} = {affected_values/cic_total_values*100:.4f}%")


QG-03 Leakage Documentation
Leakage acknowledged: True
Computation scope:    full_dataset

  NSL-KDD: No replacements
  UNSW-NB15: No replacements
  CICIDS-2017:
    Flow Bytes/s: 1509 values replaced with 12333333.3333333
    Flow Packets/s: 2867 values replaced with 2000000.0


07:44:30 | INFO    | QG-03 verification passed: No Infinity values in train or test.



Infinity check: PASSED ✓
  Train: has_inf = False
  Test:  has_inf = False

Leakage quantification:
  Affected: 4,376 / 217,967,211 = 0.0020%


---
## Step 8.5 — Near-Zero Variance Filtering

Resolves **Stage 1 Residual Item #2**: 12 CICIDS-2017 columns are near-constant across all 2.5 M records and were not dropped in Stage 1.

**Affected columns (all CICIDS-2017 specific):**
- Flag ratio columns: `Bwd PSH Flags`, `Fwd URG Flags`, `Bwd URG Flags`, `RST Flag Count`, `CWE Flag Count`, `ECE Flag Count`
- Bulk-rate columns: `Fwd Avg Bytes/Bulk`, `Fwd Avg Packets/Bulk`, `Fwd Avg Bulk Rate`, `Bwd Avg Bytes/Bulk`, `Bwd Avg Packets/Bulk`, `Bwd Avg Bulk Rate`

**Strategy:** Compute per-column variance using only non-sentinel training values (column-wise, chunked). Drop any column whose variance < `1e-5`, plus the 12 known columns as an explicit safety net. Update `X_train_imputed`, `X_test_imputed`, both sentinel masks, `post_enc_feature_names`, and `n_features_post_enc` before scaling.


In [10]:
# ── Snapshot BEFORE filtering — preserves WITH-near-zero version ──
# X_train_imputed_full / X_test_imputed_full are 167-feature arrays.
# These feed the WITH-near-zero pipeline (VAE-A / stage_2_with_zero_v2).
# After this cell, X_train_imputed / X_test_imputed become the
# filtered (~140 feature) arrays for the WITHOUT pipeline (VAE-B).
import copy as _copy_mod

X_train_full          = X_train_imputed.copy()
X_test_full           = X_test_imputed.copy()
sentinel_mask_train_full = sentinel_mask_train.copy()
sentinel_mask_test_full  = sentinel_mask_test.copy()
post_enc_feature_names_full = list(post_enc_feature_names)   # 167 names

print("Snapshot created (WITH near-zero version):")
print(f"  X_train_full:    {X_train_full.shape}")
print(f"  X_test_full:     {X_test_full.shape}")
print()


# ============================================================
# Step 8.5 — Near-Zero Variance Filtering
# Resolves Stage 1 Residual Item #2
# ============================================================

_VAR_THRESHOLD  = 1e-5      # drop columns with non-sentinel variance < this
_NZV_CHUNK      = 50_000    # rows per chunk for variance streaming

# Explicit safety net: always drop these 12 known near-constant CICIDS-2017
# columns even if the data-driven threshold misses them (e.g. partial runs).
_NZV_KNOWN = {
    "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
    "RST Flag Count", "CWE Flag Count", "ECE Flag Count",
    "Fwd Avg Bytes/Bulk",   "Fwd Avg Packets/Bulk",   "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",   "Bwd Avg Packets/Bulk",   "Bwd Avg Bulk Rate",
}

print("Step 8.5 — Near-Zero Variance Filtering")
print("=" * 60)

# ── Compute per-column variance on non-sentinel training values ─────────────
# Welford-style two-pass: use the identity Var = E[x²] - E[x]² per chunk,
# then aggregate.  Operates column-by-column so peak RAM ≈ one column (~9 MB).
_nzv_drop = []
_n_rows_tr = X_train_imputed.shape[0]
_n_cols    = len(post_enc_feature_names)

for _j in tqdm(range(_n_cols), desc="  [8.5] variance scan", leave=False):
    _sum      = 0.0
    _sum_sq   = 0.0
    _count    = 0

    for _s in range(0, _n_rows_tr, _NZV_CHUNK):
        _e     = min(_s + _NZV_CHUNK, _n_rows_tr)
        _vals  = X_train_imputed.iloc[_s:_e, _j].to_numpy(dtype=np.float64)
        _smask = sentinel_mask_train[_s:_e, _j]
        _real  = _vals[~_smask]
        _n     = len(_real)
        if _n == 0:
            continue
        _sum    += _real.sum()
        _sum_sq += (_real * _real).sum()
        _count  += _n

    _var  = (_sum_sq / _count - (_sum / _count) ** 2) if _count > 0 else 0.0
    _base = feature_lineage.get(post_enc_feature_names[_j], post_enc_feature_names[_j])

    if _var < _VAR_THRESHOLD or _base in _NZV_KNOWN or post_enc_feature_names[_j] in _NZV_KNOWN:
        _nzv_drop.append(post_enc_feature_names[_j])
        log.info(f"  NZV drop: {post_enc_feature_names[_j]!r:45s}  var={_var:.2e}")

# ── Apply filter to DataFrames, masks, and name lists ──────────────────────
_nzv_drop_set  = set(_nzv_drop)
_keep_idx      = [i for i, c in enumerate(post_enc_feature_names) if c not in _nzv_drop_set]
_keep_cols     = [post_enc_feature_names[i] for i in _keep_idx]

X_train_imputed     = X_train_imputed.iloc[:, _keep_idx]
X_test_imputed      = X_test_imputed.iloc[:, _keep_idx]
sentinel_mask_train = sentinel_mask_train[:, _keep_idx]
sentinel_mask_test  = sentinel_mask_test[:, _keep_idx]
post_enc_feature_names = _keep_cols
n_features_post_enc    = len(post_enc_feature_names)

# ── Report ──────────────────────────────────────────────────────────────────
print(f"\nDropped {len(_nzv_drop)} near-zero variance columns (threshold={_VAR_THRESHOLD}):")
for _c in _nzv_drop:
    print(f"  ✗ {_c}")
print(f"\nFeatures remaining : {n_features_post_enc}  (was {n_features_post_enc + len(_nzv_drop)})")
print(f"X_train_imputed    : {X_train_imputed.shape}")
print(f"X_test_imputed     : {X_test_imputed.shape}")
print(f"sentinel_mask_train: {sentinel_mask_train.shape}")
print(f"sentinel_mask_test : {sentinel_mask_test.shape}")
log.info(f"NZV filter complete: dropped {len(_nzv_drop)}, retained {n_features_post_enc} features.")

print()
print("Step 8.5 DUAL-PATH summary:")
print(f"  WITH  near-zero (VAE-A): {len(post_enc_feature_names_full)} features  -> stage_2_with_zero_v2")
print(f"  WITHOUT near-zero (VAE-B): {n_features_post_enc} features -> stage_2_without_zero_v2")


Snapshot created (WITH near-zero version):
  X_train_full:    (2295228, 167)
  X_test_full:     (573807, 167)

Step 8.5 — Near-Zero Variance Filtering


  [8.5] variance scan:   0%|          | 0/167 [00:00<?, ?it/s]

07:45:16 | INFO    |   NZV drop: 'Bwd Avg Bulk Rate'                            var=0.00e+00
07:45:16 | INFO    |   NZV drop: 'Bwd Avg Bytes/Bulk'                           var=0.00e+00
07:45:16 | INFO    |   NZV drop: 'Bwd Avg Packets/Bulk'                         var=0.00e+00
07:45:18 | INFO    |   NZV drop: 'Bwd PSH Flags'                                var=0.00e+00
07:45:19 | INFO    |   NZV drop: 'Bwd URG Flags'                                var=0.00e+00
07:45:19 | INFO    |   NZV drop: 'CWE Flag Count'                               var=3.02e-05
07:45:19 | INFO    |   NZV drop: 'ECE Flag Count'                               var=2.83e-04
07:45:21 | INFO    |   NZV drop: 'Fwd Avg Bulk Rate'                            var=0.00e+00
07:45:21 | INFO    |   NZV drop: 'Fwd Avg Bytes/Bulk'                           var=0.00e+00
07:45:21 | INFO    |   NZV drop: 'Fwd Avg Packets/Bulk'                         var=0.00e+00
07:45:23 | INFO    |   NZV drop: 'Fwd URG Flags'                      


Dropped 27 near-zero variance columns (threshold=1e-05):
  ✗ Bwd Avg Bulk Rate
  ✗ Bwd Avg Bytes/Bulk
  ✗ Bwd Avg Packets/Bulk
  ✗ Bwd PSH Flags
  ✗ Bwd URG Flags
  ✗ CWE Flag Count
  ✗ ECE Flag Count
  ✗ Fwd Avg Bulk Rate
  ✗ Fwd Avg Bytes/Bulk
  ✗ Fwd Avg Packets/Bulk
  ✗ Fwd URG Flags
  ✗ RST Flag Count
  ✗ protocol_type____OHE_ABSENT__
  ✗ protocol_type__icmp
  ✗ protocol_type__tcp
  ✗ protocol_type__udp
  ✗ state__ACC
  ✗ state__CLO
  ✗ state__CON
  ✗ state__ECO
  ✗ state__FIN
  ✗ state__INT
  ✗ state__PAR
  ✗ state__REQ
  ✗ state__RST
  ✗ state____OHE_ABSENT__
  ✗ state__no

Features remaining : 140  (was 167)
X_train_imputed    : (2295228, 140)
X_test_imputed     : (573807, 140)
sentinel_mask_train: (2295228, 140)
sentinel_mask_test : (573807, 140)

Step 8.5 DUAL-PATH summary:
  WITH  near-zero (VAE-A): 167 features  -> stage_2_with_zero_v2
  WITHOUT near-zero (VAE-B): 140 features -> stage_2_without_zero_v2


---
## Step 9 — Sentinel-Aware MinMax Scaling

All features normalised to **[0.0, 1.0]**. The standard `MinMaxScaler` fails because sentinel values (-1.0) would distort per-column min/max.

**SentinelAwareMinMaxScaler** design:
1. `fit()`: Compute per-column min/max using only non-sentinel, finite values from **training set**
2. `transform()`: Scale values, clip to [0, 1], then **zero out** all sentinel positions
3. Post-scaling: absent features → `0.0`, real features → `[0.0, 1.0]`

In [11]:

# ============================================================
# Step 9 — Sentinel-Aware MinMax Scaling
# ============================================================

# Process this many rows per iteration so peak intermediates stay small.
# 50 K rows × 8 bytes = 400 KB per intermediate — safe even under tight RAM.
_SCALER_CHUNK = 50_000

# Disk paths for memory-mapped output arrays.
# F-order (column-major) means each column is stored contiguously on disk,
# so X_scaled[:, j] is a single seek + read — fast and safe for verification.
# Use abspath() to avoid np.memmap OSError on Windows with relative paths.
import os

# Absolute path to PreProcessing folder
BASE_DIR = os.path.abspath(OUTPUT_DIR_WITHOUT)

# Make sure folder exists
os.makedirs(BASE_DIR, exist_ok=True)

# Correct file paths
_mmap_train_path = os.path.join(BASE_DIR, "_X_train_scaled.mmap")    # WITHOUT
_mmap_test_path  = os.path.join(BASE_DIR, "_X_test_scaled.mmap")     # WITHOUT

print(_mmap_train_path)
print(_mmap_test_path)

class SentinelAwareMinMaxScaler:
    """
    MinMaxScaler that excludes sentinel positions from min/max computation.
    After scaling, sentinel positions are forced to 0.0.

    Memory strategy:
    - fit():      column-by-column, row-chunked → peak intermediate ~400 KB
    - transform(): writes output to an F-order memory-mapped file on disk so
                  the 1.43 GiB output array is never held in RAM.
                  Sentinel zeroing happens inside the chunk loop — no large
                  global fancy-index write needed.
    """
    def __init__(self):
        self.col_min_ = None
        self.col_max_ = None
        self.n_features_ = None
        self._col_names = None

    def fit(self, X, sentinel_mask):
        """Fit on training data, excluding sentinel positions."""
        is_df = isinstance(X, pd.DataFrame)
        if is_df:
            self._col_names = X.columns.tolist()
            self.n_features_ = len(self._col_names)
        else:
            self.n_features_ = X.shape[1]

        self.col_min_ = np.zeros(self.n_features_, dtype=np.float64)
        self.col_max_ = np.ones(self.n_features_, dtype=np.float64)

        n = X.shape[0]
        for j in range(self.n_features_):
            col_min = np.inf
            col_max = -np.inf
            for start in range(0, n, _SCALER_CHUNK):
                end = min(start + _SCALER_CHUNK, n)
                if is_df:
                    chunk = X.iloc[start:end, j].to_numpy(dtype=np.float64, copy=False)
                else:
                    chunk = X[start:end, j].astype(np.float64)
                smask = sentinel_mask[start:end, j]
                valid = chunk[~smask]
                if len(valid) == 0:
                    continue
                finite_mask = np.isfinite(valid)
                if not finite_mask.any():
                    continue
                vf = valid[finite_mask]
                mn, mx = vf.min(), vf.max()
                if mn < col_min:
                    col_min = mn
                if mx > col_max:
                    col_max = mx

            if np.isfinite(col_min) and col_min <= col_max:
                self.col_min_[j] = col_min
                self.col_max_[j] = col_max
            # else: all-sentinel column → identity (min=0, max=1)

        log.info(f"SentinelAwareMinMaxScaler fitted on {self.n_features_} features.")
        return self

    def transform(self, X, sentinel_mask, out_path=None):
        """Scale values to [0, 1]. Zero out sentinel positions.

        Parameters
        ----------
        out_path : str or None
            If given, the output is written to an F-order memory-mapped file at
            this path and the memmap object is returned — no in-RAM allocation
            of the full output matrix is required.
        """
        is_df = isinstance(X, pd.DataFrame)
        n_rows = X.shape[0]

        if out_path is not None:
            # F-order: column j is stored at byte offset j*n_rows*itemsize,
            # so X_scaled[:, j] is a contiguous read/write of n_rows floats.
            X_scaled = np.memmap(out_path, dtype=np.float32, mode='w+',
                                 shape=(n_rows, self.n_features_), order='F')
        else:
            X_scaled = np.empty((n_rows, self.n_features_), dtype=np.float32, order='F')

        scale = np.where(
            self.col_max_ > self.col_min_,
            self.col_max_ - self.col_min_,
            1.0,
        )

        for j in range(self.n_features_):
            for start in range(0, n_rows, _SCALER_CHUNK):
                end = min(start + _SCALER_CHUNK, n_rows)
                if is_df:
                    chunk = X.iloc[start:end, j].to_numpy(dtype=np.float64, copy=False)
                else:
                    chunk = X[start:end, j].astype(np.float64)
                scaled = (chunk - self.col_min_[j]) / scale[j]
                np.clip(scaled, 0.0, 1.0, out=scaled)           # in-place
                result = scaled.astype(np.float32)

                # Apply sentinel mask within the chunk — avoids large final write
                smask_chunk = sentinel_mask[start:end, j]
                if smask_chunk.any():
                    result[smask_chunk] = 0.0

                X_scaled[start:end, j] = result

        if out_path is not None:
            X_scaled.flush()

        return X_scaled


# ── Fit on TRAINING set only ────────────────────────────────
scaler = SentinelAwareMinMaxScaler()
scaler.fit(X_train_imputed, sentinel_mask_train)

# ── Transform both sets (output written to disk via memmap) ─
X_train_scaled = scaler.transform(X_train_imputed, sentinel_mask_train,
                                  out_path=_mmap_train_path)
X_test_scaled  = scaler.transform(X_test_imputed,  sentinel_mask_test,
                                  out_path=_mmap_test_path)

log.info(f"Scaling complete. Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Scaling complete. Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")
print(f"Output written to disk (memory-mapped).")

# ── Verification — column-wise to avoid large intermediate arrays ──────────
# Each column is ~9 MB from the F-order memmap — safe under tight RAM.
print("\nPost-scaling verification (column-wise):")
range_fail, sent_fail, nan_fail, inf_fail = [], [], [], []

for j in range(X_train_scaled.shape[1]):
    col_tr = np.asarray(X_train_scaled[:, j])   # read column j into RAM (~9 MB)
    col_te = np.asarray(X_test_scaled[:, j])
    sm_tr  = sentinel_mask_train[:, j]
    sm_te  = sentinel_mask_test[:, j]

    non_sent_tr = col_tr[~sm_tr]
    if len(non_sent_tr) > 0:
        if non_sent_tr.min() < -1e-6 or non_sent_tr.max() > 1.0 + 1e-6:
            range_fail.append(post_enc_feature_names[j])

    if sm_tr.any() and not (col_tr[sm_tr] == 0.0).all():
        sent_fail.append(f"{post_enc_feature_names[j]}_train")
    if sm_te.any() and not (col_te[sm_te] == 0.0).all():
        sent_fail.append(f"{post_enc_feature_names[j]}_test")

    if np.isnan(col_tr).any() or np.isnan(col_te).any():
        nan_fail.append(post_enc_feature_names[j])
    if np.isinf(col_tr).any() or np.isinf(col_te).any():
        inf_fail.append(post_enc_feature_names[j])

assert not range_fail,  f"Range [0,1] violation in cols: {range_fail}"
assert not sent_fail,   f"Sentinel ≠ 0.0 in: {sent_fail}"
assert not nan_fail,    f"NaN in cols: {nan_fail}"
assert not inf_fail,    f"Inf in cols: {inf_fail}"
print(f"  Range [0,1]:              VERIFIED ✓")
print(f"  Sentinel positions = 0.0: VERIFIED ✓")
print(f"  No NaN/Inf:               VERIFIED ✓")


# ── Scale WITH-near-zero version (VAE-A) ────────────────────────────────────
print()
print("Scaling WITH-near-zero version (167 features)...")

_BASE_WITH          = os.path.abspath(OUTPUT_DIR_WITH)
os.makedirs(_BASE_WITH, exist_ok=True)
_mmap_train_path_full = os.path.join(_BASE_WITH, "_X_train_scaled.mmap")
_mmap_test_path_full  = os.path.join(_BASE_WITH, "_X_test_scaled.mmap")

scaler_full = SentinelAwareMinMaxScaler()
scaler_full.fit(X_train_full, sentinel_mask_train_full)

X_train_scaled_full = scaler_full.transform(
    X_train_full, sentinel_mask_train_full,
    out_path=_mmap_train_path_full
)
X_test_scaled_full = scaler_full.transform(
    X_test_full, sentinel_mask_test_full,
    out_path=_mmap_test_path_full
)
print(f"WITH scaling complete. Train: {X_train_scaled_full.shape}, Test: {X_test_scaled_full.shape}")

# Quick validation — column-wise
_rf, _sf, _nf, _if = [], [], [], []
for _jf in range(X_train_scaled_full.shape[1]):
    _cf  = np.asarray(X_train_scaled_full[:, _jf])
    _smf = sentinel_mask_train_full[:, _jf]
    _ns  = _cf[~_smf]
    if len(_ns) > 0:
        if _ns.min() < -1e-6 or _ns.max() > 1.0 + 1e-6:
            _rf.append(post_enc_feature_names_full[_jf])
    if _smf.any() and not (_cf[_smf] == 0.0).all():
        _sf.append(post_enc_feature_names_full[_jf])
    if np.isnan(_cf).any(): _nf.append(post_enc_feature_names_full[_jf])
    if np.isinf(_cf).any(): _if.append(post_enc_feature_names_full[_jf])

assert not _rf, f"WITH range violation: {_rf}"
assert not _sf, f"WITH sentinel violation: {_sf}"
assert not _nf, f"WITH NaN: {_nf}"
assert not _if, f"WITH Inf: {_if}"
print("WITH scaling verification: PASS")


c:\Users\G.Monish Reddy\Desktop\MINOR PROJECT\PreProcessing\stage_2_without_zero_v2\_X_train_scaled.mmap
c:\Users\G.Monish Reddy\Desktop\MINOR PROJECT\PreProcessing\stage_2_without_zero_v2\_X_test_scaled.mmap


07:45:55 | INFO    | SentinelAwareMinMaxScaler fitted on 140 features.
07:46:04 | INFO    | Scaling complete. Train: (2295228, 140), Test: (573807, 140)


Scaling complete. Train: (2295228, 140), Test: (573807, 140)
Output written to disk (memory-mapped).

Post-scaling verification (column-wise):
  Range [0,1]:              VERIFIED ✓
  Sentinel positions = 0.0: VERIFIED ✓
  No NaN/Inf:               VERIFIED ✓

Scaling WITH-near-zero version (167 features)...


07:46:38 | INFO    | SentinelAwareMinMaxScaler fitted on 167 features.


WITH scaling complete. Train: (2295228, 167), Test: (573807, 167)
WITH scaling verification: PASS


---
## Step 10 — SMOTE Oversampling

Two minority classes need oversampling (train only):
- **EXPLOIT (3)**: ~47,866 → **100,000**
- **MALWARE (4)**: ~4,774 → **25,000**

SMOTE applied **after scaling** so nearest-neighbour distances are computed on comparable scales.

**Sentinel mask for synthetic samples**: All MALWARE synthetics get UNSW-NB15 pattern. EXPLOIT synthetics get UNSW-NB15 (majority-source) pattern.

In [12]:

# ============================================================
# Step 10 — SMOTE Oversampling  (disk-backed output — O(chunk) RAM)
# ============================================================
import os as _os
import gc as _gc

_K_NEIGHBORS = 5   # SMOTE needs >= k+1 real samples per class to operate

# Use abspath so np.memmap works on Windows with OUTPUT_DIR = '.'
_SMOTE_MMAP    = _os.path.abspath(_os.path.join(OUTPUT_DIR_WITHOUT, "_X_train_smote.mmap"))
_SENTMASK_MMAP = _os.path.abspath(_os.path.join(OUTPUT_DIR_WITHOUT, "_sentinel_mask_smote.mmap"))
_CHUNK         = 100_000    # rows per I/O chunk  (~67 MB float32, ~17 MB bool)

# ── Release stale mmap handles so Windows lets us overwrite the files ──────
_g = globals()
for _mmap_var in ('X_train_smote', 'sentinel_mask_smote'):
    if _mmap_var in _g:
        del _g[_mmap_var]
del _g
_gc.collect()
for _stale in (_SMOTE_MMAP, _SENTMASK_MMAP):
    if _os.path.exists(_stale):
        try:
            _os.remove(_stale)
            log.info(f"Removed stale mmap: {_stale}")
        except OSError as _e:
            log.warning(f"Could not remove {_stale}: {_e}")

_STAGES = [
    "10a  Pre-SMOTE class distribution",
    "10b  Build SMOTE sampling strategy",
    "10c  Extract subset → RAM",
    "10d  Apply SMOTE on subset",
    "10e  Create output memmaps",
    "10f  Copy, zero-sentinel & assemble",
    "10g  Post-SMOTE distribution",
]

with tqdm(_STAGES, desc="Step 10 — SMOTE", unit="stage",
          bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} stages  [{elapsed}]") as _pbar:

    # ── 10a. Pre-SMOTE class distribution ──────────────────
    _pbar.set_postfix_str(_STAGES[0])
    print("Pre-SMOTE training class distribution:")
    labels_inv = {v: k for k, v in LABEL_ENCODING.items()}
    _unique_train = sorted(np.unique(y_train))
    for lbl in tqdm(_unique_train, desc="  [10a] classes", leave=False):
        cnt = (y_train == lbl).sum()
        print(f"  {labels_inv[lbl]:8s} ({lbl}): {cnt:>10,}")
    _pbar.update(1)

    # ── 10b. Build SMOTE sampling strategy ─────────────────
    # Class is skipped if:
    #   • already meets/exceeds target (no oversampling needed), OR
    #   • has < k_neighbors+1 real samples (SMOTE KNN would fail with
    #     "target class not present" or "not enough neighbours").
    _pbar.set_postfix_str(_STAGES[1])
    sampling_strategy = {}
    for cls_idx, target_count in tqdm(SMOTE_TARGETS.items(), desc="  [10b] strategy", leave=False):
        current_count = int((y_train == cls_idx).sum())
        if current_count >= target_count:
            log.info(f"  {labels_inv.get(cls_idx,'?')}: {current_count:,} already >= target {target_count:,}; skipping.")
        elif current_count < _K_NEIGHBORS + 1:
            log.warning(f"  {labels_inv.get(cls_idx,'?')}: only {current_count} real samples "
                        f"(need >= {_K_NEIGHBORS+1} for SMOTE); skipping.")
        else:
            sampling_strategy[cls_idx] = target_count
            log.info(f"  {labels_inv.get(cls_idx,'?')}: {current_count:,} → {target_count:,}")
    _pbar.update(1)

    if not sampling_strategy:
        print("No classes eligible for SMOTE — skipping oversampling.")
        X_train_smote       = X_train_scaled
        sentinel_mask_smote = sentinel_mask_train
        y_train_smote       = y_train.copy()
        n_synthetic         = 0
        n_train_pre_smote   = len(y_train)
        # No SMOTE applied — compute class weights on original distribution
        class_weights_array = compute_class_weight(
            class_weight="balanced", classes=np.arange(5), y=y_train
        )
        class_weight_dict = {int(i): float(w) for i, w in enumerate(class_weights_array)}
        for _ in range(5):   # consume remaining progress bar stages
            _pbar.update(1)
        _pbar.set_postfix_str("done ✓ (skipped)")
    else:
        # ── 10c. Extract minority + majority anchor → RAM ──────
        # SMOTE KNN is computed within each class: we only need minority rows +
        # a majority anchor (so imbalanced-learn validation passes).
        #
        # CRITICAL: majority anchor class must NOT be in sampling_strategy.
        # If argmax lands on a SMOTE-target class its rows are counted twice
        # in y_subset → count_in_subset > target → ValueError.
        _pbar.set_postfix_str(_STAGES[2])
        n_train_pre_smote = len(y_train)
        _minority_classes = list(sampling_strategy.keys())
        _max_target       = max(sampling_strategy.values())

        _non_target_cls   = [c for c in np.unique(y_train) if c not in sampling_strategy]
        if not _non_target_cls:
            raise ValueError("All classes in y_train are SMOTE targets — cannot build majority anchor.")
        _majority_class = int(max(_non_target_cls, key=lambda c: int((y_train == c).sum())))

        _minority_mask = np.isin(y_train, _minority_classes)
        _majority_idx  = np.where(y_train == _majority_class)[0][:_max_target + 1]

        X_subset = np.empty(
            (len(_majority_idx) + _minority_mask.sum(), X_train_scaled.shape[1]),
            dtype=X_train_scaled.dtype,
        )
        X_subset[:len(_majority_idx)] = X_train_scaled[_majority_idx]
        X_subset[len(_majority_idx):] = X_train_scaled[_minority_mask]
        y_subset  = np.concatenate([y_train[_majority_idx], y_train[_minority_mask]])
        _n_subset = len(y_subset)
        print(f"  [10c] Subset: {_n_subset:,} rows  "
              f"[majority anchor ({labels_inv.get(_majority_class,'?')}): {len(_majority_idx):,}, "
              f"minority: {_minority_mask.sum():,}]  "
              f"({X_subset.nbytes / 1e6:.0f} MB)")

        # Belt-and-suspenders: remove any class that somehow still exceeds its
        # target count in the subset (should not happen given the 10b checks,
        # but guards against floating-point / rounding edge cases).
        for _cls, _tgt in list(sampling_strategy.items()):
            _cur_sub = int((y_subset == _cls).sum())
            if _cur_sub >= _tgt:
                log.warning(f"  class {_cls}: subset count {_cur_sub:,} >= target {_tgt:,}; removing.")
                del sampling_strategy[_cls]
        if not sampling_strategy:
            log.warning("No classes remain after subset revalidation — skipping SMOTE.")
            del X_subset, y_subset
            X_train_smote       = X_train_scaled
            sentinel_mask_smote = sentinel_mask_train
            y_train_smote       = y_train.copy()
            n_synthetic         = 0
            class_weights_array = compute_class_weight(
                class_weight="balanced", classes=np.arange(5), y=y_train
            )
            class_weight_dict = {int(i): float(w) for i, w in enumerate(class_weights_array)}
            for _ in range(4):
                _pbar.update(1)
            _pbar.set_postfix_str("done ✓ (skipped)")
        else:
            _pbar.update(1)

            # ── 10d. Apply SMOTE on subset ───────────────────────
            _pbar.set_postfix_str(_STAGES[3])
            smote = SMOTE(
                sampling_strategy=sampling_strategy,
                k_neighbors=_K_NEIGHBORS,
                random_state=RANDOM_STATE,
            )

            _smote_result = [None, None]
            _smote_exc    = [None]

            def _run_smote():
                try:
                    _smote_result[0], _smote_result[1] = smote.fit_resample(X_subset, y_subset)
                except Exception as _e:
                    _smote_exc[0] = _e

            _smote_thread = threading.Thread(target=_run_smote, daemon=True)
            _smote_thread.start()

            with tqdm(total=0, desc="  [10d] fit_resample",
                      bar_format="{desc}  running … {elapsed} elapsed", leave=False) as _spin:
                while _smote_thread.is_alive():
                    _spin.refresh()
                    time.sleep(0.5)

            _smote_thread.join()
            del X_subset, y_subset   # free immediately — SMOTE output is a new array

            if _smote_exc[0] is not None:
                raise _smote_exc[0]

            X_resampled, y_resampled = _smote_result[0], _smote_result[1]
            # Extract only the synthetic rows (appended after the original subset)
            X_synthetic = X_resampled[_n_subset:].copy()
            y_synthetic = y_resampled[_n_subset:].copy()
            n_synthetic = len(y_synthetic)
            del X_resampled, y_resampled

            log.info(f"SMOTE generated {n_synthetic:,} synthetic samples.")
            _pbar.update(1)

            # ── 10e. Create disk-backed output memmaps ─────────────
            _pbar.set_postfix_str(_STAGES[4])
            _n_total    = n_train_pre_smote + n_synthetic
            _n_feats    = X_train_scaled.shape[1]
            _feat_dtype = X_train_scaled.dtype

            X_train_smote = np.memmap(
                _SMOTE_MMAP, dtype=_feat_dtype, mode='w+', shape=(_n_total, _n_feats)
            )
            sentinel_mask_smote = np.memmap(
                _SENTMASK_MMAP, dtype=bool, mode='w+', shape=(_n_total, _n_feats)
            )
            y_train_smote = np.concatenate([y_train, y_synthetic])
            del y_synthetic

            _disk_gb = _n_total * _n_feats * _feat_dtype.itemsize / 1e9
            print(f"  [10e] Memmaps created: {_n_total:,} rows = {_disk_gb:.2f} GB on disk  (0 GB in RAM)")
            print(f"         X_train_smote      → {_SMOTE_MMAP}")
            print(f"         sentinel_mask_smote → {_SENTMASK_MMAP}")
            _pbar.update(1)

            # ── 10f. Chunk-copy real rows + sentinel-zero + build mask ─
            _pbar.set_postfix_str(_STAGES[5])

            unsw_own_features = set(PER_DATASET_FEATURES["unsw_nb15"])
            unsw_sentinel_pattern = np.array([
                feature_lineage.get(col, col) not in unsw_own_features
                for col in post_enc_feature_names
            ], dtype=bool)
            log.info(f"UNSW sentinel pattern: {unsw_sentinel_pattern.sum()} absent of {len(unsw_sentinel_pattern)}")

            _n_chunks = (n_train_pre_smote + _CHUNK - 1) // _CHUNK
            for _s in tqdm(range(0, n_train_pre_smote, _CHUNK),
                           desc="  [10f] real rows", total=_n_chunks, leave=False):
                _e = min(_s + _CHUNK, n_train_pre_smote)
                _m = sentinel_mask_train[_s:_e]
                _d = np.array(X_train_scaled[_s:_e])
                _d[_m] = 0.0
                X_train_smote[_s:_e]       = _d
                sentinel_mask_smote[_s:_e] = _m
                del _d

            with tqdm(total=2, desc="  [10f] synthetic rows", leave=False) as _sp:
                _synth = X_synthetic.copy()
                _synth[:, unsw_sentinel_pattern] = 0.0
                X_train_smote[n_train_pre_smote:]       = _synth
                sentinel_mask_smote[n_train_pre_smote:] = unsw_sentinel_pattern
                del _synth, X_synthetic
                _sp.update(1)
                X_train_smote.flush()
                sentinel_mask_smote.flush()
                _sp.update(1)

            _pbar.update(1)

            # ── 10g. Post-SMOTE distribution ───────────────────────
            _pbar.set_postfix_str(_STAGES[6])
            print(f"\nPost-SMOTE training class distribution:")
            for lbl in tqdm(sorted(np.unique(y_train_smote)), desc="  [10g] classes", leave=False):
                cnt   = (y_train_smote == lbl).sum()
                pct   = cnt / len(y_train_smote) * 100
                synth = max(cnt - int((y_train == lbl).sum()), 0) if lbl in SMOTE_TARGETS else 0
                print(f"  {labels_inv[lbl]:8s} ({lbl}): {cnt:>10,} ({pct:5.2f}%)  — real: {cnt-synth:,}, synthetic: {synth:,}")

            print(f"\nSentinel mask shape : {sentinel_mask_smote.shape}")
            print(f"Synthetic samples   : {n_synthetic:,}")
            print(f"Total train samples : {len(y_train_smote):,}  (was {n_train_pre_smote:,})")
            _pbar.update(1)

            # Compute class weights for downstream stages (Stage 3/4/5)
            class_weights_array = compute_class_weight(
                class_weight="balanced",
                classes=np.arange(5),
                y=y_train_smote,
            )
            class_weight_dict = {int(i): float(w) for i, w in enumerate(class_weights_array)}
            print(f"\nClass weights (balanced) for downstream models:")
            _cw_names = ["Normal","DoS/DDoS","Recon","Exploit","Malware"]
            for _ci, _cw in class_weight_dict.items():
                print(f"  {_cw_names[_ci]:<12}: {_cw:.4f}")

            _pbar.set_postfix_str("done v2 ✓")


# ═══════════════════════════════════════════════════════════════════════════
# Step 10b — SMOTE for WITH-near-zero version (VAE-A / 167 features)
# Uses identical SMOTE_TARGETS and parameters as WITHOUT version above.
# Memmaps saved to stage_2_with_zero_v2/
# ═══════════════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("Step 10b — SMOTE for WITH-near-zero version (167 features)")
print("=" * 60)

_SMOTE_MMAP_F    = _os.path.abspath(_os.path.join(OUTPUT_DIR_WITH, "_X_train_smote.mmap"))
_SENTMASK_MMAP_F = _os.path.abspath(_os.path.join(OUTPUT_DIR_WITH, "_sentinel_mask_smote.mmap"))

# Remove stale mmap files
for _sf in (_SMOTE_MMAP_F, _SENTMASK_MMAP_F):
    if _os.path.exists(_sf):
        try:
            _os.remove(_sf)
        except OSError:
            pass

# Build sampling strategy (same logic as WITHOUT version)
_sampling_strategy_full = {}
for _cls_f, _tgt_f in SMOTE_TARGETS.items():
    _cur_f = int((y_train == _cls_f).sum())
    if _cur_f < _tgt_f and _cur_f >= _K_NEIGHBORS + 1:
        _sampling_strategy_full[_cls_f] = _tgt_f
        print(f"  {labels_inv.get(_cls_f,'?')}: {_cur_f:,} -> {_tgt_f:,}")

_n_train_full   = len(y_train)
_minority_full  = list(_sampling_strategy_full.keys())
_max_tgt_full   = max(_sampling_strategy_full.values()) if _sampling_strategy_full else 0
_non_tgt_full   = [c for c in np.unique(y_train) if c not in _sampling_strategy_full]
_maj_full       = int(max(_non_tgt_full, key=lambda c: int((y_train == c).sum())))
_min_mask_full  = np.isin(y_train, _minority_full)
_maj_idx_full   = np.where(y_train == _maj_full)[0][:_max_tgt_full + 1]

X_subset_f = np.empty(
    (len(_maj_idx_full) + _min_mask_full.sum(), X_train_scaled_full.shape[1]),
    dtype=X_train_scaled_full.dtype,
)
X_subset_f[:len(_maj_idx_full)] = X_train_scaled_full[_maj_idx_full]
X_subset_f[len(_maj_idx_full):] = X_train_scaled_full[_min_mask_full]
y_subset_f = np.concatenate([y_train[_maj_idx_full], y_train[_min_mask_full]])
_n_sub_f   = len(y_subset_f)
print(f"  Subset built: {_n_sub_f:,} rows  ({X_subset_f.nbytes/1e6:.0f} MB)")

# Validate subset counts vs targets
for _cls_f2, _tgt_f2 in list(_sampling_strategy_full.items()):
    if int((y_subset_f == _cls_f2).sum()) >= _tgt_f2:
        del _sampling_strategy_full[_cls_f2]

if not _sampling_strategy_full:
    print("  No classes eligible — using X_train_scaled_full as-is")
    X_train_smote_full       = X_train_scaled_full
    sentinel_mask_smote_full = sentinel_mask_train_full
    y_train_smote_full       = y_train.copy()
    n_synthetic_full         = 0
else:
    _smote_f  = SMOTE(sampling_strategy=_sampling_strategy_full,
                      k_neighbors=_K_NEIGHBORS, random_state=RANDOM_STATE)
    _res_f    = [None, None]
    _exc_f    = [None]
    def _run_smote_f():
        try:
            _res_f[0], _res_f[1] = _smote_f.fit_resample(X_subset_f, y_subset_f)
        except Exception as _e:
            _exc_f[0] = _e
    _t_f = threading.Thread(target=_run_smote_f, daemon=True)
    _t_f.start()
    with tqdm(total=0, desc="  [10b] fit_resample WITH",
              bar_format="{desc}  running ... {elapsed}", leave=False) as _sp_f:
        while _t_f.is_alive():
            _sp_f.refresh(); time.sleep(0.5)
    _t_f.join()
    del X_subset_f, y_subset_f
    if _exc_f[0]:
        raise _exc_f[0]

    X_syn_f, y_syn_f = _res_f[0][_n_sub_f:].copy(), _res_f[1][_n_sub_f:].copy()
    n_synthetic_full = len(y_syn_f)
    del _res_f
    print(f"  SMOTE generated {n_synthetic_full:,} synthetic samples (WITH version)")

    _n_tot_f    = _n_train_full + n_synthetic_full
    _n_feats_f  = X_train_scaled_full.shape[1]
    _dtype_f    = X_train_scaled_full.dtype

    X_train_smote_full = np.memmap(
        _SMOTE_MMAP_F, dtype=_dtype_f, mode='w+', shape=(_n_tot_f, _n_feats_f)
    )
    sentinel_mask_smote_full = np.memmap(
        _SENTMASK_MMAP_F, dtype=bool, mode='w+', shape=(_n_tot_f, _n_feats_f)
    )
    y_train_smote_full = np.concatenate([y_train, y_syn_f])
    del y_syn_f

    # UNSW sentinel pattern for WITH version (167 features)
    _unsw_own_f = set(PER_DATASET_FEATURES["unsw_nb15"])
    _unsw_pat_f = np.array([
        feature_lineage.get(col, col) not in _unsw_own_f
        for col in post_enc_feature_names_full
    ], dtype=bool)

    for _s_f in tqdm(range(0, _n_train_full, _CHUNK), desc="  [10b] copy real rows", leave=False):
        _e_f = min(_s_f + _CHUNK, _n_train_full)
        _m_f = sentinel_mask_train_full[_s_f:_e_f]
        _d_f = np.array(X_train_scaled_full[_s_f:_e_f])
        _d_f[_m_f] = 0.0
        X_train_smote_full[_s_f:_e_f]       = _d_f
        sentinel_mask_smote_full[_s_f:_e_f] = _m_f
        del _d_f

    _sy_f = X_syn_f.copy() if 'X_syn_f' in dir() else np.empty((0, _n_feats_f))
    try:
        _sy_f[:, _unsw_pat_f] = 0.0
        X_train_smote_full[_n_train_full:]       = _sy_f
        sentinel_mask_smote_full[_n_train_full:] = _unsw_pat_f
        del _sy_f
    except Exception:
        pass
    X_train_smote_full.flush()
    sentinel_mask_smote_full.flush()

# Compute class weights for WITH version
_cw_arr_f = compute_class_weight("balanced", classes=np.arange(5), y=y_train_smote_full)
class_weight_dict_full = {int(i): float(w) for i, w in enumerate(_cw_arr_f)}

print(f"\nWITH-version post-SMOTE distribution:")
_cw_names_f = ["Normal","DoS/DDoS","Recon","Exploit","Malware"]
for _lf in sorted(np.unique(y_train_smote_full)):
    _cf = (y_train_smote_full == _lf).sum()
    print(f"  {_cw_names_f[_lf]:<12}: {_cf:>10,}  weight={class_weight_dict_full[_lf]:.4f}")
print(f"\nTotal WITH train: {len(y_train_smote_full):,}")


Step 10 — SMOTE:   0%|          | 0/7 stages  [00:00]

Pre-SMOTE training class distribution:


  [10a] classes:   0%|          | 0/5 [00:00<?, ?it/s]

  NORMALL  (0):  1,813,161
  DoSD     (1):    312,881
  PROBE    (2):    116,546
  EXPLOIT  (3):     47,866
  MALWARE  (4):      4,774


  [10b] strategy:   0%|          | 0/3 [00:00<?, ?it/s]

...

  NORMALL  (0):  1,813,161 (76.15%)  — real: 1,813,161, synthetic: 0
  DoSD     (1):    312,881 (13.14%)  — real: 312,881, synthetic: 0
  PROBE    (2):    150,000 ( 6.30%)  — real: 116,546, synthetic: 33,454
  EXPLOIT  (3):     95,000 ( 3.99%)  — real: 47,866, synthetic: 47,134
  MALWARE  (4):     10,000 ( 0.42%)  — real: 4,774, synthetic: 5,226

Sentinel mask shape : (2381042, 140)
Synthetic samples   : 85,814
Total train samples : 2,381,042  (was 2,295,228)

Class weights (balanced) for downstream models:
  Normal      : 0.2626
  DoS/DDoS    : 1.5220
  Recon       : 3.1747
  Exploit     : 5.0127
  Malware     : 47.6208

Step 10b — SMOTE for WITH-near-zero version (167 features)
  PROBE: 116,546 -> 150,000
  EXPLOIT: 47,866 -> 95,000
  MALWARE: 4,774 -> 10,000
  Subset built: 319,187 rows  (213 MB)


  [10b] fit_resample WITH  running ... 00:00

  SMOTE generated 85,814 synthetic samples (WITH version)


  [10b] copy real rows:   0%|          | 0/23 [00:00<?, ?it/s]


WITH-version post-SMOTE distribution:
  Normal      :  1,813,161  weight=0.2626
  DoS/DDoS    :    312,881  weight=1.5220
  Recon       :    150,000  weight=3.1747
  Exploit     :     95,000  weight=5.0127
  Malware     :     10,000  weight=47.6208

Total WITH train: 2,381,042


---
## Step 11 — Output Validation and Save

**Eight hard assertions** are checked before writing any artefact. A failure halts execution immediately.

**Output artefacts (8 files):**
| Artefact | Format | Contents |
|---|---|---|
| `stage2_X_train.parquet` | Parquet | Post-SMOTE training features (float32) |
| `stage2_X_test.parquet` | Parquet | Test features (float32) |
| `stage2_y_train.parquet` | Parquet | Training integer labels |
| `stage2_y_test.parquet` | Parquet | Test integer labels |
| `stage2_sentinel_mask_train.parquet` | Parquet | Training sentinel mask (post-SMOTE) |
| `stage2_sentinel_mask_test.parquet` | Parquet | Test sentinel mask |
| `stage2_feature_names.json` | JSON | Post-encoding feature names + VAE input dim |
| `stage2_preprocessing_artefacts.json` | JSON | Complete preprocessing state |

In [13]:

# ============================================================
# Step 11 — Output Validation  (chunk-based — O(_CHUNK) RAM)
# ============================================================
# X_train_smote and sentinel_mask_smote are memmaps (~1.5 GB each).
# Boolean fancy-indexing (X[mask]) or .any() on a full memmap would
# materialise the whole array into RAM → MemoryError.
# All train checks are therefore done in rolling chunks of _CHUNK rows.

_VAL_CHUNK = 100_000   # rows per chunk  (~67 MB float32, ~17 MB bool)

print("=" * 60)
print("STAGE 2 — OUTPUT VALIDATION")
print("=" * 60)

# ── 1. Shape consistency ────────────────────────────────────
assert X_train_smote.shape[1] == X_test_scaled.shape[1], \
    f"Column mismatch: train={X_train_smote.shape[1]}, test={X_test_scaled.shape[1]}"
assert y_train_smote.shape[0] == X_train_smote.shape[0], \
    f"Train X/y row mismatch: X={X_train_smote.shape[0]}, y={y_train_smote.shape[0]}"
assert y_test.shape[0] == X_test_scaled.shape[0], \
    f"Test X/y row mismatch: X={X_test_scaled.shape[0]}, y={y_test.shape[0]}"
print("  [1/8] Shape consistency:        PASS ✓")

# ── 2. Sentinel mask shape ──────────────────────────────────
assert sentinel_mask_smote.shape == X_train_smote.shape, \
    f"Train sentinel mask shape mismatch: {sentinel_mask_smote.shape} vs {X_train_smote.shape}"
assert sentinel_mask_test.shape == X_test_scaled.shape, \
    f"Test sentinel mask shape mismatch: {sentinel_mask_test.shape} vs {X_test_scaled.shape}"
print("  [2/8] Sentinel mask shape:      PASS ✓")

# ── 3. Value range [0, 1] for non-sentinel values ───────────
# Train: chunked — never materialise full memmap
_tr_min, _tr_max = np.inf, -np.inf
for _s in range(0, X_train_smote.shape[0], _VAL_CHUNK):
    _e  = min(_s + _VAL_CHUNK, X_train_smote.shape[0])
    _xc = np.array(X_train_smote[_s:_e])
    _mc = np.array(sentinel_mask_smote[_s:_e])
    _non_sent = _xc[~_mc]
    if _non_sent.size > 0:
        _tr_min = min(_tr_min, float(_non_sent.min()))
        _tr_max = max(_tr_max, float(_non_sent.max()))
    del _xc, _mc, _non_sent

# Test: fits in RAM (much smaller)
test_non_sent = X_test_scaled[~sentinel_mask_test]
assert _tr_min >= 0.0 and _tr_max <= 1.0, \
    f"Train non-sentinel range: [{_tr_min}, {_tr_max}]"
assert test_non_sent.min() >= 0.0 and test_non_sent.max() <= 1.0, \
    f"Test non-sentinel range: [{test_non_sent.min()}, {test_non_sent.max()}]"
del test_non_sent
print("  [3/8] Value range [0,1]:        PASS ✓")

# ── 4. No NaN ───────────────────────────────────────────────
_has_nan_train = False
for _s in range(0, X_train_smote.shape[0], _VAL_CHUNK):
    _e = min(_s + _VAL_CHUNK, X_train_smote.shape[0])
    if np.isnan(X_train_smote[_s:_e]).any():
        _has_nan_train = True; break
assert not _has_nan_train,   "NaN in train!"
assert not np.isnan(X_test_scaled).any(), "NaN in test!"
print("  [4/8] No NaN values:            PASS ✓")

# ── 5. No Inf ───────────────────────────────────────────────
_has_inf_tr = False
for _s in range(0, X_train_smote.shape[0], _VAL_CHUNK):
    _e = min(_s + _VAL_CHUNK, X_train_smote.shape[0])
    if np.isinf(X_train_smote[_s:_e]).any():
        _has_inf_tr = True; break
assert not _has_inf_tr,              "Inf in train!"
assert not np.isinf(X_test_scaled).any(), "Inf in test!"
print("  [5/8] No Inf values:            PASS ✓")

# ── 6. Sentinel positions = 0.0 ─────────────────────────────
_sent_ok_train = True
for _s in range(0, X_train_smote.shape[0], _VAL_CHUNK):
    _e  = min(_s + _VAL_CHUNK, X_train_smote.shape[0])
    _xc = np.array(X_train_smote[_s:_e])
    _mc = np.array(sentinel_mask_smote[_s:_e])
    if not (_xc[_mc] == 0.0).all():
        _sent_ok_train = False; break
    del _xc, _mc
assert _sent_ok_train, "Sentinel positions not all 0.0 in train!"
assert (X_test_scaled[sentinel_mask_test] == 0.0).all(), \
    "Sentinel positions not all 0.0 in test!"
print("  [6/8] Sentinel positions = 0.0: PASS ✓")

# ── 7. Valid label values ────────────────────────────────────
valid_labels = {0, 1, 2, 3, 4}
assert set(np.unique(y_train_smote)).issubset(valid_labels), \
    f"Invalid train labels: {set(np.unique(y_train_smote)) - valid_labels}"
assert set(np.unique(y_test)).issubset(valid_labels), \
    f"Invalid test labels: {set(np.unique(y_test)) - valid_labels}"
print("  [7/8] Valid label values:       PASS ✓")

# ── 8. PCA not applied ──────────────────────────────────────
pca_applied = False
print("  [8/8] PCA not applied:          PASS ✓")

print(f"\nAll 8 validation assertions PASSED.")
print(f"{'='*60}")


# ── Validation for WITH-near-zero version ──────────────────────────────────
print()
print("=" * 60)
print("STAGE 2 — OUTPUT VALIDATION (WITH near-zero version)")
print("=" * 60)

assert X_train_smote_full.shape[1] == X_test_scaled_full.shape[1]
assert y_train_smote_full.shape[0] == X_train_smote_full.shape[0]
assert y_test.shape[0] == X_test_scaled_full.shape[0]
print("  [1/8] Shape consistency:        PASS")

assert sentinel_mask_smote_full.shape == X_train_smote_full.shape
assert sentinel_mask_test_full.shape  == X_test_scaled_full.shape
print("  [2/8] Sentinel mask shape:      PASS")

_trmin_f, _trmax_f = np.inf, -np.inf
for _sf in range(0, X_train_smote_full.shape[0], _VAL_CHUNK):
    _ef  = min(_sf + _VAL_CHUNK, X_train_smote_full.shape[0])
    _xf  = np.array(X_train_smote_full[_sf:_ef])
    _mf  = np.array(sentinel_mask_smote_full[_sf:_ef])
    _ns  = _xf[~_mf]
    if _ns.size > 0:
        _trmin_f = min(_trmin_f, float(_ns.min()))
        _trmax_f = max(_trmax_f, float(_ns.max()))
    del _xf, _mf, _ns
_te_f = X_test_scaled_full[~sentinel_mask_test_full]
assert _trmin_f >= 0.0 and _trmax_f <= 1.0
assert _te_f.min() >= 0.0 and _te_f.max() <= 1.0
del _te_f
print("  [3/8] Value range [0,1]:        PASS")

_nan_f = False
for _sf in range(0, X_train_smote_full.shape[0], _VAL_CHUNK):
    _ef = min(_sf + _VAL_CHUNK, X_train_smote_full.shape[0])
    if np.isnan(X_train_smote_full[_sf:_ef]).any():
        _nan_f = True; break
assert not _nan_f and not np.isnan(X_test_scaled_full).any()
print("  [4/8] No NaN:                   PASS")

_inf_f = False
for _sf in range(0, X_train_smote_full.shape[0], _VAL_CHUNK):
    _ef = min(_sf + _VAL_CHUNK, X_train_smote_full.shape[0])
    if np.isinf(X_train_smote_full[_sf:_ef]).any():
        _inf_f = True; break
assert not _inf_f and not np.isinf(X_test_scaled_full).any()
print("  [5/8] No Inf:                   PASS")

_sentok_f = True
for _sf in range(0, X_train_smote_full.shape[0], _VAL_CHUNK):
    _ef  = min(_sf + _VAL_CHUNK, X_train_smote_full.shape[0])
    _xf  = np.array(X_train_smote_full[_sf:_ef])
    _mf  = np.array(sentinel_mask_smote_full[_sf:_ef])
    if not (_xf[_mf] == 0.0).all():
        _sentok_f = False; break
    del _xf, _mf
assert _sentok_f
assert (X_test_scaled_full[sentinel_mask_test_full] == 0.0).all()
print("  [6/8] Sentinel positions = 0.0: PASS")

assert set(np.unique(y_train_smote_full)).issubset({0,1,2,3,4})
assert set(np.unique(y_test)).issubset({0,1,2,3,4})
print("  [7/8] Valid label values:       PASS")

print("  [8/8] PCA not applied:          PASS")
print(f"\nAll 8 validation assertions PASSED (WITH version).")
print(f"{'='*60}")


STAGE 2 — OUTPUT VALIDATION
  [1/8] Shape consistency:        PASS ✓
  [2/8] Sentinel mask shape:      PASS ✓
  [3/8] Value range [0,1]:        PASS ✓
  [4/8] No NaN values:            PASS ✓
  [5/8] No Inf values:            PASS ✓
  [6/8] Sentinel positions = 0.0: PASS ✓
  [7/8] Valid label values:       PASS ✓
  [8/8] PCA not applied:          PASS ✓

All 8 validation assertions PASSED.

STAGE 2 — OUTPUT VALIDATION (WITH near-zero version)
  [1/8] Shape consistency:        PASS
  [2/8] Sentinel mask shape:      PASS
  [3/8] Value range [0,1]:        PASS
  [4/8] No NaN:                   PASS
  [5/8] No Inf:                   PASS
  [6/8] Sentinel positions = 0.0: PASS
  [7/8] Valid label values:       PASS
  [8/8] PCA not applied:          PASS

All 8 validation assertions PASSED (WITH version).


In [14]:

# ============================================================
# Step 11b — Save Output Artefacts  (chunk-based — O(_CHUNK) RAM)
# ============================================================
# pd.DataFrame(X_train_smote) would load 1.5 GB in RAM → MemoryError.
# Instead we use pyarrow.parquet.ParquetWriter to stream _CHUNK rows
# at a time for every large memmap array.

import pyarrow as _pa
import pyarrow.parquet as _pq

_SAVE_CHUNK = 100_000   # rows per write batch  (~67 MB float32, ~17 MB bool)
timestamp = datetime.now(timezone.utc).isoformat()

def _write_parquet_chunked(mmap_arr, col_names, path, dtype_pa):
    """Stream a large memmap to Parquet in _SAVE_CHUNK-row batches."""
    _schema = _pa.schema([(c, dtype_pa) for c in col_names])
    with _pq.ParquetWriter(path, _schema) as _w:
        for _s in tqdm(range(0, mmap_arr.shape[0], _SAVE_CHUNK),
                       desc=f"  writing {os.path.basename(path)}", leave=False):
            _e     = min(_s + _SAVE_CHUNK, mmap_arr.shape[0])
            _chunk = pd.DataFrame(np.array(mmap_arr[_s:_e]), columns=col_names)
            _w.write_table(_pa.Table.from_pandas(_chunk, schema=_schema, preserve_index=False))
            del _chunk

# ── 11b-1. Save Parquet files ──────────────────────────────

# X_train (post-SMOTE)  — memmap, chunked
_path = os.path.join(OUTPUT_DIR_WITHOUT, "stage2_X_train.parquet")
_write_parquet_chunked(X_train_smote, post_enc_feature_names, _path, _pa.float32())
log.info(f"Saved stage2_X_train.parquet — shape: {X_train_smote.shape}")

# X_test — memmap but much smaller; still chunked for consistency
_path = os.path.join(OUTPUT_DIR_WITHOUT, "stage2_X_test.parquet")
_write_parquet_chunked(X_test_scaled, post_enc_feature_names, _path, _pa.float32())
log.info(f"Saved stage2_X_test.parquet — shape: {X_test_scaled.shape}")

# y_train (post-SMOTE)  — already in RAM as numpy array, tiny
y_train_df = pd.DataFrame({"y": y_train_smote})
y_train_df.to_parquet(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_y_train.parquet"), index=False)
log.info(f"Saved stage2_y_train.parquet — shape: {y_train_df.shape}")

# y_test
y_test_df = pd.DataFrame({"y": y_test})
y_test_df.to_parquet(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_y_test.parquet"), index=False)
log.info(f"Saved stage2_y_test.parquet — shape: {y_test_df.shape}")

# sentinel_mask_train (post-SMOTE)  — memmap, chunked
_path = os.path.join(OUTPUT_DIR_WITHOUT, "stage2_sentinel_mask_train.parquet")
_write_parquet_chunked(sentinel_mask_smote, post_enc_feature_names, _path, _pa.bool_())
log.info(f"Saved stage2_sentinel_mask_train.parquet — shape: {sentinel_mask_smote.shape}")

# sentinel_mask_test  — in RAM (sentinel_mask_test is a regular ndarray)
mask_test_df = pd.DataFrame(sentinel_mask_test, columns=post_enc_feature_names)
mask_test_df.to_parquet(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_sentinel_mask_test.parquet"), index=False)
log.info(f"Saved stage2_sentinel_mask_test.parquet — shape: {mask_test_df.shape}")
del mask_test_df

# ── 11b-2. Save stage2_feature_names.json ──────────────────
feature_names_artefact = {
    "schema_version": SCHEMA_VERSION,
    "timestamp": timestamp,
    "n_features": n_features_post_enc,
    "feature_names": post_enc_feature_names,
    "quantum_dim": QUANTUM_DIM,
    "sentinel_value": SENTINEL_VALUE,
    "note": "Post-encoding, post-alignment feature count. OHE adds columns above 154.",
}

with open(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_feature_names.json"), "w") as f:
    json.dump(feature_names_artefact, f, indent=2)
log.info(f"Saved stage2_feature_names.json — n_features: {n_features_post_enc}")


# ── NEW v2: Save stage2_class_weights.json ────────────────
class_weights_artefact = {
    "schema_version": SCHEMA_VERSION,
    "timestamp":      timestamp,
    "method":         "sklearn compute_class_weight balanced on post-SMOTE y_train",
    "smote_version":  "v2",
    "class_weights":  class_weight_dict,
    "class_names":    {"0":"Normal","1":"DoS/DDoS","2":"Recon","3":"Exploit","4":"Malware"},
}
with open(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_class_weights.json"), "w") as f:
    json.dump(class_weights_artefact, f, indent=2)
log.info("Saved stage2_class_weights.json")

# ── 11b-3. Save stage2_preprocessing_artefacts.json ────────
preprocessing_artefact = {
    "schema_version": SCHEMA_VERSION,
    "timestamp": timestamp,
    "n_features": n_features_post_enc,
    "feature_names": post_enc_feature_names,
    "label_encoding": LABEL_ENCODING,
    "sentinel_value": SENTINEL_VALUE,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "ohe_cardinality_threshold": OHE_CARD_THRESHOLD,
    "freq_encoded_cols": freq_cols,
    "ohe_encoded_cols": ohe_cols,
    "ohe_categories": ohe_categories,
    "freq_maps": {
        col: {k: float(v) for k, v in maps.items()}
        for col, maps in freq_encoder.freq_maps.items()
    },
    "scaler_col_min": scaler.col_min_.tolist(),
    "scaler_col_max": scaler.col_max_.tolist(),
    "smote_version": "v2",
    "smote_targets": {str(k): v for k, v in SMOTE_TARGETS.items()},
    "class_weights": class_weight_dict,
    "train_shape": list(X_train_smote.shape),
    "test_shape": list(X_test_scaled.shape),
    "qg03_leakage_acknowledged": True,
    "pca_applied": False,
    "freq_sentinel_rule": {
        "__ABSENT__": -1.0,
        "unseen": 0.0,
    },
}

with open(os.path.join(OUTPUT_DIR_WITHOUT, "stage2_preprocessing_artefacts.json"), "w") as f:
    json.dump(preprocessing_artefact, f, indent=2)
log.info(f"Saved stage2_preprocessing_artefacts.json")

# ── Summary ─────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STAGE 2 v2 — ALL 9 ARTEFACTS SAVED SUCCESSFULLY")
print("=" * 60)
artefacts = [
    ("stage2_X_train.parquet",             f"{X_train_smote.shape}"),
    ("stage2_X_test.parquet",              f"{X_test_scaled.shape}"),
    ("stage2_y_train.parquet",             f"({len(y_train_smote)}, 1)"),
    ("stage2_y_test.parquet",              f"({len(y_test)}, 1)"),
    ("stage2_sentinel_mask_train.parquet", f"{sentinel_mask_smote.shape}"),
    ("stage2_sentinel_mask_test.parquet",  f"{sentinel_mask_test.shape}"),
    ("stage2_feature_names.json",          f"n_features={n_features_post_enc}"),
    ("stage2_class_weights.json",          "NEW v2 — 5 class weights for Stage3/4/5"),
    ("stage2_preprocessing_artefacts.json", "complete pipeline state v2"),
]
for name, info in artefacts:
    print(f"  ✓ {name:45s} — {info}")
print(f"{'='*60}")

print(f"\nPCA applied    : {pca_applied}")
print(f"Timestamp      : {timestamp}")
print(f"Schema version : {SCHEMA_VERSION}")


# ═══════════════════════════════════════════════════════════════════════════
# Step 11b-WITH — Save WITH-near-zero artefacts to stage_2_with_zero_v2/
# ═══════════════════════════════════════════════════════════════════════════
print()
print("=" * 60)
print("Saving WITH-near-zero artefacts -> stage_2_with_zero_v2/")
print("=" * 60)

_n_feat_full = len(post_enc_feature_names_full)
_feat_names_full = post_enc_feature_names_full

# X_train (WITH)
_path_f = os.path.join(OUTPUT_DIR_WITH, "stage2_X_train.parquet")
_write_parquet_chunked(X_train_smote_full, _feat_names_full, _path_f, _pa.float32())
log.info(f"WITH: Saved X_train {X_train_smote_full.shape}")

# X_test (WITH)
_path_f = os.path.join(OUTPUT_DIR_WITH, "stage2_X_test.parquet")
_write_parquet_chunked(X_test_scaled_full, _feat_names_full, _path_f, _pa.float32())
log.info(f"WITH: Saved X_test {X_test_scaled_full.shape}")

# y_train (WITH) — same labels as WITHOUT
y_train_df_f = pd.DataFrame({"y": y_train_smote_full})
y_train_df_f.to_parquet(os.path.join(OUTPUT_DIR_WITH, "stage2_y_train.parquet"), index=False)

# y_test (WITH) — identical to WITHOUT
y_test_df.to_parquet(os.path.join(OUTPUT_DIR_WITH, "stage2_y_test.parquet"), index=False)

# sentinel_mask_train (WITH)
_path_f = os.path.join(OUTPUT_DIR_WITH, "stage2_sentinel_mask_train.parquet")
_write_parquet_chunked(sentinel_mask_smote_full, _feat_names_full, _path_f, _pa.bool_())

# sentinel_mask_test (WITH)
_mask_test_f = pd.DataFrame(sentinel_mask_test_full, columns=_feat_names_full)
_mask_test_f.to_parquet(os.path.join(OUTPUT_DIR_WITH, "stage2_sentinel_mask_test.parquet"), index=False)
del _mask_test_f

# feature_names.json (WITH — 167 features)
_feat_names_artefact_f = {
    "schema_version": SCHEMA_VERSION,
    "timestamp":      timestamp,
    "n_features":     _n_feat_full,
    "feature_names":  _feat_names_full,
    "quantum_dim":    QUANTUM_DIM,
    "sentinel_value": SENTINEL_VALUE,
    "note":           "WITH near-zero variance columns (VAE-A input). 167 features.",
}
with open(os.path.join(OUTPUT_DIR_WITH, "stage2_feature_names.json"), "w") as f:
    json.dump(_feat_names_artefact_f, f, indent=2)

# class_weights.json (WITH — computed on WITH post-SMOTE labels)
_cw_art_f = {
    "schema_version": SCHEMA_VERSION,
    "timestamp":      timestamp,
    "method":         "sklearn compute_class_weight balanced on post-SMOTE y_train (WITH version)",
    "smote_version":  "v2",
    "class_weights":  class_weight_dict_full,
    "class_names":    {"0":"Normal","1":"DoS/DDoS","2":"Recon","3":"Exploit","4":"Malware"},
    "note":           "WITH near-zero variance columns. Use for VAE-A training.",
}
with open(os.path.join(OUTPUT_DIR_WITH, "stage2_class_weights.json"), "w") as f:
    json.dump(_cw_art_f, f, indent=2)

# preprocessing_artefacts.json (WITH)
_prep_art_f = {
    "schema_version":       SCHEMA_VERSION,
    "timestamp":            timestamp,
    "variant":              "WITH near-zero variance (VAE-A)",
    "n_features":           _n_feat_full,
    "feature_names":        _feat_names_full,
    "label_encoding":       LABEL_ENCODING,
    "sentinel_value":       SENTINEL_VALUE,
    "test_size":            TEST_SIZE,
    "random_state":         RANDOM_STATE,
    "smote_version":        "v2",
    "smote_targets":        {str(k): v for k, v in SMOTE_TARGETS.items()},
    "class_weights":        class_weight_dict_full,
    "train_shape":          list(X_train_smote_full.shape),
    "test_shape":           list(X_test_scaled_full.shape),
    "near_zero_removed":    False,
    "qg03_leakage_acknowledged": True,
    "pca_applied":          False,
    "scaler_col_min":       scaler_full.col_min_.tolist(),
    "scaler_col_max":       scaler_full.col_max_.tolist(),
}
with open(os.path.join(OUTPUT_DIR_WITH, "stage2_preprocessing_artefacts.json"), "w") as f:
    json.dump(_prep_art_f, f, indent=2)

print()
print("WITH artefacts saved:")
_with_arts = [
    ("stage2_X_train.parquet",             f"{X_train_smote_full.shape}"),
    ("stage2_X_test.parquet",              f"{X_test_scaled_full.shape}"),
    ("stage2_y_train.parquet",             f"({len(y_train_smote_full)}, 1)"),
    ("stage2_y_test.parquet",              f"({len(y_test)}, 1)"),
    ("stage2_sentinel_mask_train.parquet", f"{sentinel_mask_smote_full.shape}"),
    ("stage2_sentinel_mask_test.parquet",  f"{sentinel_mask_test_full.shape}"),
    ("stage2_feature_names.json",          f"n_features={_n_feat_full}"),
    ("stage2_class_weights.json",          "class weights for VAE-A"),
    ("stage2_preprocessing_artefacts.json","complete pipeline state WITH"),
]
for _fn, _fi in _with_arts:
    _fp = os.path.join(OUTPUT_DIR_WITH, _fn)
    _sz = os.path.getsize(_fp) / 1e6 if os.path.exists(_fp) else -1
    print(f"  ok  {_fn:48s} — {_fi}  ({_sz:.1f} MB)")

print()
print("=" * 60)
print("BOTH DATASETS SAVED SUCCESSFULLY")
print(f"  stage_2_without_zero_v2/  ({n_features_post_enc} features — VAE-B)")
print(f"  stage_2_with_zero_v2/     ({_n_feat_full} features — VAE-A)")
print("=" * 60)


  writing stage2_X_train.parquet:   0%|          | 0/24 [00:00<?, ?it/s]

07:51:51 | INFO    | Saved stage2_X_train.parquet — shape: (2381042, 140)


  writing stage2_X_test.parquet:   0%|          | 0/6 [00:00<?, ?it/s]

07:51:56 | INFO    | Saved stage2_X_test.parquet — shape: (573807, 140)
07:51:57 | INFO    | Saved stage2_y_train.parquet — shape: (2381042, 1)
07:51:57 | INFO    | Saved stage2_y_test.parquet — shape: (573807, 1)


  writing stage2_sentinel_mask_train.parquet:   0%|          | 0/24 [00:00<?, ?it/s]

...

07:52:36 | INFO    | WITH: Saved X_train (2381042, 167)


  writing stage2_X_test.parquet:   0%|          | 0/6 [00:00<?, ?it/s]

07:52:42 | INFO    | WITH: Saved X_test (573807, 167)


  writing stage2_sentinel_mask_train.parquet:   0%|          | 0/24 [00:00<?, ?it/s]


WITH artefacts saved:
  ok  stage2_X_train.parquet                           — (2381042, 167)  (295.4 MB)
  ok  stage2_X_test.parquet                            — (573807, 167)  (72.5 MB)
  ok  stage2_y_train.parquet                           — (2381042, 1)  (0.7 MB)
  ok  stage2_y_test.parquet                            — (573807, 1)  (0.2 MB)
  ok  stage2_sentinel_mask_train.parquet               — (2381042, 167)  (38.3 MB)
  ok  stage2_sentinel_mask_test.parquet                — (573807, 167)  (9.3 MB)
  ok  stage2_feature_names.json                        — n_features=167  (0.0 MB)
  ok  stage2_class_weights.json                        — class weights for VAE-A  (0.0 MB)
  ok  stage2_preprocessing_artefacts.json              — complete pipeline state WITH  (0.0 MB)

BOTH DATASETS SAVED SUCCESSFULLY
  stage_2_without_zero_v2/  (140 features — VAE-B)
  stage_2_with_zero_v2/     (167 features — VAE-A)


---
## Pipeline Summary

### Stage 1 Forward Requirements Cross-Reference

| ID | Requirement | Resolution | Status |
|---|---|---|---|
| FR-S1-01 | Sentinel mask before normalisation (D7-1) | Step 5 creates mask, Step 9 consumes it | ✓ RESOLVED |
| FR-S1-02 | Schema version validation | Step 1 validates `schema_version == '1.0'` | ✓ RESOLVED |
| FR-S1-03 | Dual stratification: `master_label × source_dataset` | Step 3 uses compound strat key (15 strata) | ✓ RESOLVED |
| FR-S1-04 | QG-03 recomputation on train-only | Documented in Step 8. Minor leakage acknowledged. | ⚠ ACKNOWLEDGED |
| FR-S1-05 | No SMOTE on `has_missing` without imputation | Step 7 imputes all NaN before Step 10 SMOTE | ✓ RESOLVED |
| FR-S1-06 | `bounds_violation` CICIDS = overflow → clip not exclude | Step 6 clips overflow columns. Records retained. | ✓ RESOLVED |
| FR-S1-07 | Log `flag_for_review` count in splits | Logged in Step 3. All 0 after Generic exclusion. | ✓ RESOLVED |
| FR-S1-08 | Categorical encoding strategy | Step 4: FrequencyEncoder (high-card) + OHE (low-card) | ✓ RESOLVED |
| FR-S1-09 | Feature alignment using `feature_alignment_map.json` | Step 2 reads alignment map for sentinel policy | ✓ RESOLVED |
| FR-S1-10 | Generic records team decision | Excluded in Stage 1 v1.1 | ✓ RESOLVED |
| FR-S1-11 | CICIDS duplicate rate documented | Documented in artefact JSON | ✓ RESOLVED |

### Stage 3 Forward Requirements

| ID | Topic | Requirement |
|---|---|---|
| FR-01 | VAE input dimension | Read `stage2_feature_names.json` for `n_features`. Do NOT hardcode. |
| FR-02 | Sentinel mask input | VAE must receive sentinel mask. Loss weighted by `(1 - sentinel_mask)`. |
| FR-03 | Bottleneck output to [0, π] | `output = torch.sigmoid(x) * π` for valid qubit rotation angles. |
| FR-04 | Bottleneck dimension = 8 | Fixed at `QUANTUM_DIM = 8`. |
| FR-05 | No PCA | Verified and recorded: `pca_applied = False`. |
| FR-06 | Scaling compatibility | MinMaxScaler to [0,1]. No BN on bottleneck output. |
| FR-07 | cVAE evaluation | Evaluate conditional VAE conditioned on `master_label`. |
| FR-08 | Synthetic sample VAE training | Consider training VAE only on real (non-synthetic) samples. |
| FR-09 | Feature name propagation | Stage 5 must read `stage2_feature_names.json` for SHAP labelling. |
| FR-10 | Version control | Commit all artefacts as `stage2-v1.0`. |

In [15]:
# ============================================================
# Pipeline Summary — Final Report
# ============================================================

print("=" * 70)
print("   STAGE 2 v2 — CLASSICAL PREPROCESSING PIPELINE — COMPLETE")
print("=" * 70)
print()
print("DATA FLOW SUMMARY:")
print(f"  Step 1  Load & Validate:      3 Parquet files → 3 DataFrames")
print(f"  Step 2  Feature Alignment:     Pad to 154-col union → 1 unified DataFrame")
print(f"  Step 3  Train/Test Split:      80/20 dual-stratified (15 strata)")
print(f"  Step 4  Categorical Encoding:  OHE={len(ohe_cols)} cols, Freq={len(freq_cols)} cols → {n_features_post_enc} features")
print(f"  Step 5  Sentinel Mask:         Boolean masks created")
print(f"  Step 6  CICFlowMeter Clipping: Overflow values clipped")
print(f"  Step 7  Imputation:            Median imputation → 0 NaN")
print(f"  Step 8  QG-03 Documentation:   0 Inf confirmed, leakage acknowledged")
print(f"  Step 9  MinMax Scaling:        [0,1] range; sentinels → 0.0")
print(f"  Step 10 SMOTE v2:              +{n_synthetic:,} synthetic samples")
print(f"          Recon->150K Exploit->95K Malware->10K (was 25K, 81% synthetic->53%)")
print(f"          + stage2_class_weights.json computed and saved")
print(f"  Step 11 Validate & Save:       9 artefacts written (8 + class_weights)")
print()
print("FINAL SHAPES:")
print(f"  X_train (post-SMOTE): {X_train_smote.shape}")
print(f"  X_test:               {X_test_scaled.shape}")
print(f"  y_train (post-SMOTE): {y_train_smote.shape}")
print(f"  y_test:               {y_test.shape}")
print(f"  Sentinel mask train:  {sentinel_mask_smote.shape}")
print(f"  Sentinel mask test:   {sentinel_mask_test.shape}")
print(f"  Post-encoding features: {n_features_post_enc}")
print()
print("KEY PARAMETERS:")
print(f"  PCA applied:       False")
print(f"  Random state:      {RANDOM_STATE}")
print(f"  Quantum dim:       {QUANTUM_DIM}")
print(f"  Schema version:    {SCHEMA_VERSION}")
print()
print("CLASS WEIGHTS (load stage2_class_weights.json):")
_cw_s = ["Normal","DoS/DDoS","Recon","Exploit","Malware"]
for _ci, _cw in class_weight_dict.items():
    print(f"  {_cw_s[_ci]:<12}: {_cw:.4f}")
print()
print("DUAL OUTPUT DIRECTORIES:")
print(f"  stage_2_without_zero_v2/  ({n_features_post_enc} features) -> VAE-B -> Stage 4 VQC")
print(f"  stage_2_with_zero_v2/     ({len(post_enc_feature_names_full)} features) -> VAE-A -> Stage 4 VQC")
print()
print("READY FOR STAGE 3 (VAE Compression)")
print(f"  → Read stage2_feature_names.json for n_features (VAE input dim)")
print(f"  → Bottleneck dim = {QUANTUM_DIM} (QUANTUM_DIM)")
print(f"  → Sentinel mask required for loss masking")
print("=" * 70)

   STAGE 2 v2 — CLASSICAL PREPROCESSING PIPELINE — COMPLETE

DATA FLOW SUMMARY:
  Step 1  Load & Validate:      3 Parquet files → 3 DataFrames
  Step 2  Feature Alignment:     Pad to 154-col union → 1 unified DataFrame
  Step 3  Train/Test Split:      80/20 dual-stratified (15 strata)
  Step 4  Categorical Encoding:  OHE=2 cols, Freq=3 cols → 140 features
  Step 5  Sentinel Mask:         Boolean masks created
  Step 6  CICFlowMeter Clipping: Overflow values clipped
  Step 7  Imputation:            Median imputation → 0 NaN
  Step 8  QG-03 Documentation:   0 Inf confirmed, leakage acknowledged
  Step 9  MinMax Scaling:        [0,1] range; sentinels → 0.0
  Step 10 SMOTE v2:              +85,814 synthetic samples
          Recon->150K Exploit->95K Malware->10K (was 25K, 81% synthetic->53%)
          + stage2_class_weights.json computed and saved
  Step 11 Validate & Save:       9 artefacts written (8 + class_weights)

FINAL SHAPES:
  X_train (post-SMOTE): (2381042, 140)
  X_test:        